# FIR Document Image Preprocessing

Step-by-step cleanup of scanned FIR case files using **OpenCV + NumPy only** (no third-party AI or OCR).
Each step is one cell: it records per-page metrics, the actions it took, and review flags. The full plan is in `README.md`.

## Step 0: Setup & configuration

- `CONFIG`: every parameter and review threshold in one place. Each later step adds its own section.
- `device.py`: detects the machine (Mac M5 / NVIDIA GB10 / generic CPU) and returns the settings profile (backends, parallel workers, memory budget).
- Input PDFs are found and one **page record** is created per page. Every later step writes into these records.

In [ ]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pypdfium2 as pdfium
import pypdfium2.raw as pdfium_raw

from device import apply_profile, get_device_profile, print_profile

%config InlineBackend.figure_format = 'retina'

In [ ]:
CONFIG = {
    "input": {
        "pdf_dir": "sample-docs",
        "pdf_files": ["FIR-2-2009.pdf"],        # None = every PDF in pdf_dir, or a list such as ["FIR-2-2009.pdf"]
        "page_range": None,       # None = all pages, or (first, last) 1-based inclusive, e.g. (1, 3) for quick tests
    },
    "output": {
        "dir": "output",          # results go to output/<doc>/
        "png_compression": 3,     # 0-9 (higher = smaller, slower)
        "bw": {                   # black & white version (Sauvola adaptive threshold)
            "source": "min_channel",  # "min_channel" (darkest of B,G,R: faint blue ink is much darker in red) or
                                      # "luminance". Measured: min_channel + k 0.15 keeps 98.8% of stroke cores
                                      # (median) vs 92.7% for luminance + k 0.2, with B&W noise still <= 0.15%
            "window_mm": 6.0,     # local window (a few text-line heights)
            "k": 0.15,            # Sauvola k: higher = thinner/lighter strokes
            "R": 128,             # dynamic range of the standard deviation
            "min_blob_mm2": 0.02, # isolated black blobs smaller than this are removed
        },
        "save_stages": None,      # None = follow PROFILE["save_intermediates"]; True/False to force
        "page_digits": 3,         # {doc}_{page}_...: page number zero-padded to at least this many digits
    },
    "batch": {                    # streaming mode for large document sets (see "Batch mode" cell)
        "pages_in_flight": None,  # pages held in memory at once; None = 2 x PROFILE["num_workers"]
        "keep_images": False,     # True keeps every page's images in memory (only for small sets / review)
    },
    "render": {
        "upscale": 1.0,           # >1 enlarges after extraction (small handwriting); capped by PROFILE["max_render_scale"]
        "fallback_dpi": 200,      # only for pages that are not a single full-page scan image
        "assumed_paper_width_in": 8.5,  # PDF says 72 DPI (meaningless); estimate DPI from the short side
        "dpi_override": None,     # set a number if the estimate is wrong (all physical sizes below scale with DPI)
        "dpi_per_document": True, # one scanner session per PDF: use the median page estimate (narrow sheets
                                  # otherwise look low-DPI because the estimate assumes full-width paper)
        "color_scale": 0.25,      # colour stats on a 1/4-size copy
        "color_chroma_min": 40,   # max-min channel difference that counts as a "coloured" pixel
        "color_paper_min": 150,   # pixels brighter than this (all channels) count as paper for the tint
        "greyscale_max_ink_pct": 0.05,  # a scan is "greyscale" if coloured pixels < this % ...
        "greyscale_max_tint": 3,        # ... and paper chroma < this
    },
    "assess": {
        # All sizes are in millimetres and converted per page with its DPI (Step 1),
        # so a 150 / 300 DPI scan gets the same physical behaviour as the 200 DPI samples.
        "work_scale": 4,                # borders/illumination measured on a 1/4-size copy
        "reference_dpi": 200,           # page content is resampled to this DPI before measuring, so pixel-level
                                        # metrics (sharpness, noise, specks) mean the same for any scan resolution
        # --- borders ---
        "paper_ref_percentile": 50,     # paper colour = median of the brightest half of the page centre
        "border_dark_rel": 0.40,        # "dark" = darker than 40% of this page's paper level
        "border_dark_frac": 0.35,       # a line belongs to a dark strip if >= 35% of it is dark
        "border_dark_min_mm": 1.5,      # a dark strip must be at least this wide (table rules are thinner)
        "border_dark_grow_frac": 0.15,  # then grow the strip inward while lines are still >= 15% dark (shadow edge) ...
        "border_dark_grow_max_mm": 3.0, # ... by at most this much
        "border_pale_min_mm": 1.0,      # minimum width of a pale scanner-bed strip
        "border_dark_search": 0.20,     # search for dark strips in the outer 20% of each side
        "border_pale_search": 0.08,     # search for pale strips in the outer 8%
        "border_pale_de": 12,           # pale strip: Lab colour distance from paper >= this ...
        "border_pale_std": 12,          # ... and uniform along the line (lightness std < this; higher = text)
        # --- strokes (black-hat = how much darker than the local background) ---
        "stroke_width_max_mm": 1.9,     # strokes thinner than this count as ink
        "stroke_weak": 12, "stroke_ink": 30, "stroke_strong": 45,  # black-hat grey levels
        "contrast_percentile": 90,      # stroke contrast = this percentile of black-hat on ink
        # --- sharpness / noise ---
        "sharp_sigma_mm": 0.13,         # high-frequency = image - Gaussian(this)
        "sharp_band_mm": 0.38,          # measure on a band this wide around strokes
        "noise_flat_percentile": 90,    # Immerkaer noise on pixels below this gradient percentile
        # --- faint marks (bleed-through vs texture) ---
        "rule_line_min_mm": 7.7,        # straight runs longer than this are ruled/dotted lines
        "rule_line_gap_mm": 1.1,        # gaps bridged when finding dotted lines
        "rule_line_pad_mm": 0.6,        # margin removed around found lines
        "near_ink_mm": 1.1,             # faint marks closer than this to real ink are stroke edges, not bleed
        "faint_stroke_min_area_mm2": 0.65,  # faint component must be at least this big ...
        "faint_stroke_min_size_mm": 1.0,    # ... at least this tall AND wide (curvy, not a line) ...
        "faint_stroke_max_fill": 0.6,       # ... and fill < 60% of its box (stroke, not a blob)
        "paper_pad_mm": 0.6,            # paper pixels = this far from any mark
        # --- specks ---
        "speck_max_area_mm2": 0.10,     # blobs up to this size can be specks
        "speck_isolation_mm": 2.1,      # ... if no other ink in a window this wide
        # --- illumination ---
        "illum_close_mm": 7.6,          # closing removes strokes before estimating the paper level
        "illum_median_mm": 10.7,
        "illum_percentiles": (5, 95),
        # --- ink colours (OpenCV hue 0-180; red wraps around 180) ---
        "ink_chroma_min": 30,
        "ink_hues": {"blue": (90, 130), "purple": (130, 165), "red": (165, 12)},
        # --- what later steps should do ("needs") ---
        "needs": {
            "sharpen_below": 7.0,       # sharpness score -> Step 8
            "denoise_mild_sigma": 1.0,  # noise sigma -> Step 7 (mild)
            "denoise_strong_sigma": 2.0,  # -> Step 7 (strong)
            "denoise_texture_pct": 2.0, # faint dotty texture -> Step 7 (mild)
            "speck_removal_per_mp": 100,  # isolated specks / megapixel -> Step 7
            "bleed_faint_stroke_pct": 1.0,  # faint stroke-like marks -> Step 9
            "illumination_spread": 25,  # paper-level spread -> Step 6
            "paper_tint": 6,            # paper chroma -> Step 6 tint removal
            "faint_ink_level": 180,     # median ink grey level -> Step 10 contrast
        },
        # --- unknown-issue guard: range of each metric seen on the 71 sample pages (with margin).
        # A page outside a range is flagged "unusual_<metric>" for review: it may have a problem we
        # have not designed for. Widen a range only after looking at the page.
        "expected_ranges": {
            "ink_pct": (0.3, 55), "sharpness": (4.5, 20), "noise_sigma": (0, 4.5), "texture_pct": (0, 3.5),
            "isolated_specks_per_mp": (0, 250), "faint_stroke_pct": (0, 2.5), "illumination_spread": (0, 80),
            "paper_tint": (0, 22), "paper_level": (150, 255), "ink_level": (25, 200), "content_frac": (0.7, 1.0),
        },
    },
    "orient": {
        # Text mask reuses CONFIG["assess"] stroke + ruled-line settings (mm, scaled per page).
        "work_dpi": 150,                # orientation measured on a 150 DPI copy of the content box
        # 1) Axis: smear text along x and along y; horizontal text gives long flat blobs when smeared along x
        "smear_mm": 3.0,
        "smear_elongation": 4.0,        # blob counts as a "line" if length >= 4x thickness
        "axis_confident": 0.10,         # |axis score| below this = not sure which way the text runs
        # 2) Up/down: horizontal bars (Devanagari headline, T-bars) have ink hanging BELOW them when upright
        "bar_min_mm": 2.0,              # horizontal strokes at least this long count as bars
        "bar_window_mm": 2.0,           # compare ink this far above vs below each bar
        "updown_rotate_below": -0.5,    # vote <= this: rotate 180 (no upright sample went below -0.40)
        "updown_flag_below": -0.15,     # vote <= this: flag "possibly_upside_down" for review (English pages
                                        # give weak/negative votes; the vote is reliable for Hindi text)
        "min_text_px": 2000,            # fewer text pixels (at work_dpi) = too little text to decide
    },
    "borders": {
        "fill": "paper",                # "paper" = this page's paper colour (Step 2), or "white"
        "protect_mm": 1.5,              # ink within this distance inside the content box that crosses into a band is kept
        "restore_pad_mm": 0.25,         # kept strokes get this much extra margin (anti-aliased edges)
        "restore_reach_mm": 4.0,        # a kept stroke may reach at most this far into the band (descender, loop)
        "restore_max_stroke_mm2": 60,   # connected ink bigger than this is shadow/texture, not a pen stroke
        "restore_max_stroke_width_mm": 0.8,  # thicker than this (widest point) is a blob, not a pen stroke
        "restore_min_background_rel": 0.75,  # only restore strokes lying on paper-bright background (>= 75% of the
                                             # page's paper level); strokes inside a dark spine shadow are not text
        "feather_mm": 0.5,              # soft blend from the kept page into the filled band
        # Dark (spine/shadow) bands are FLATTENED, not painted over: page text often lies on the shadow.
        "shadow_smooth_mm": 0.4,        # blur speckle before estimating the shadow level
        "shadow_bg_mm": 3.8,            # background = closing wider than any stroke (after the blur)
        "shadow_max_gain": 6.0,         # background darker than paper/6 is too dark to flatten ...
        "shadow_lift": None,            # None (default, reviewed and chosen): flattened shadow only, no data loss.
                                        # (0.45, 0.80) = optional MONOTONE tone curve inside dark bands: lightness below
                                        # 45% of paper kept (text), above 80% -> paper (grain), smooth between; also
                                        # reviewed as acceptable. (0.40, 0.70) was judged too strong.
                                        # Binary "text or not" masks were tried and erased characters.
        "shadow_lift_feather_mm": 1.0,  # soft edge where the band meets the page
        "fill_unrecoverable_dark": False,  # ... and is LEFT AS ORIGINAL (False, safe: may hold white-on-black text or
                                           # text the eye can still read). True paints it with paper (cosmetic only).
        "shadow_mask_smooth_mm": 2.0,   # smooth the "unrecoverable" mask so the fill edge is not jagged
        "shadow_grain_rel": 0.8,        # after flattening, blurred level >= 80% of paper = grain, not ink -> paper
        "shadow_grain_blur_mm": 0.3,    # (0.7 / 0.4 mm erased faint text on the shadow; leftover grain -> Step 7)
        "crease_smooth_mm": 1.0,        # profile smoothing when locating the spine
        # White-on-dark print (black banners with white text) inside a dark band must not be flattened/filled
        "inverted_bg_rel": 0.35,        # pixels darker than 35% of paper are "black" ...
        "inverted_solid_mm": 4.0,       # ... a white-on-black area is one where >= 60% of a 4 mm window is black
        "inverted_solid_frac": 0.6,     #     (gaps between dark letters on paper are NOT a dark background) ...
        "inverted_contrast_rel": 0.5,   # ... and marks brighter than it by >= 50% of paper = white-on-black glyphs
                                        # (spine shadows: grey ~40-50% of paper, gaps only ~25% brighter -> no)
        "inverted_min_glyph_mm2": 0.3,  # bright marks at least this big ...
        "inverted_glyph_size_mm": (0.5, 15.0),  # ... with min side >= 0.5 mm and max side <= 15 mm ...
        "inverted_max_elongation": 6.0, # ... and not line-like (crease highlights)
        "inverted_min_glyphs": 5,       # this many glyph-like marks in a dark band -> flag "inverted_content_in_border"
        "shadow_edge_rel": 0.5,         # the shadow's outer edge = where the profile rises back above
                                        # min + 50% of (paper - min) on the edge side of the darkest line.
                                        # Area beyond it may be a neighbour-page sliver OR this page continuing
                                        # (lines written across the spine): it is flattened, never erased, and
                                        # flagged "content_beyond_spine" if it holds ink.
    },
    "deskew": {
        "work_dpi": 150,                # skew measured on a 150 DPI copy of the content box
        "max_angle": 10.0,              # search range +- degrees
        "coarse_step": 0.5, "fine_step": 0.05,  # projection search: coarse, then fine around the best
        "min_text_px": 2000,            # fewer text pixels at work_dpi = no projection estimate
        # printed / dotted lines (Hough)
        "line_min_mm": 30,              # segments at least this long
        "line_gap_mm": 1.5,             # bridge gaps (dotted fill-in lines)
        "line_votes_mm": 20,            # Hough accumulator threshold (in mm of collinear ink)
        "min_lines": 3,                 # fewer segments = no line estimate
        # decision (thresholds from the self-test on tilted sample pages)
        "lines_strong_min": 10,         # >= 10 segments and ...
        "lines_strong_mm": 1000,        # ... >= 1 m total length: median error 0.09 deg, 95% within 0.5 deg
        "projection_strong_peak": 1.1,  # text-row sharpness at best angle / at 0 deg; below this it was often wrong
        "agree_deg": 0.5,               # estimates within this agree
        "projection_conflict_deg": 1.0, # projection decides, but even weak line evidence differs by more -> uncertain
        "min_rotate_deg": 0.1,          # smaller skew is left alone
        "uncertain_flag_deg": 0.5,      # no trustworthy estimate but one suggests >= this -> flag
    },
    "illumination": {
        "apply_to_all_pages": True,     # True: every non-blank page gets the same white paper (pages that are already
                                        # white change by ~nothing); False: only pages Step 2 says need it
        "bg_dpi": 50,                   # paper colour estimated on a 50 DPI copy
        "bg_window_mm": 12.0,           # closing window: wider than strokes, dense handwriting gaps, banner bars
        "bg_smooth_mm": 12.0,           # median + Gaussian smoothing of the estimate (no halos)
        "target": 255,                  # paper becomes this (white)
        "max_gain": 1.5,                # never brighten more than this: dark areas wider than the window
                                        # (thumbprints, big stamps) are not bleached
        "min_gain": 0.95,               # and never darken by more than 5%
        "snap_start": 245,              # soft white-point: luminance from here to 255 is eased toward pure white
                                        # (was 235: faded pencil marks and very light stamps)
        "dark_object_rel": 0.55,        # estimated "paper" darker than 55% of the page's paper = an OBJECT (photo,
                                        # thumbprint, seal, black banner), not paper: the paper estimate there is
                                        # filled in from the surrounding paper so the object is not brightened
    },
    "denoise": {
        # Decided from metrics re-measured on the current image (Step 4 flattening can add grain after Step 2).
        # Level thresholds reuse CONFIG["assess"]["needs"] (denoise_mild_sigma / denoise_strong_sigma / ...).
        "nlm_h": {"mild": 6, "strong": 12},   # non-local-means strength on lightness (0-255 scale)
        "nlm_template_mm": 0.9,         # patch size
        "nlm_search_mm": 2.7,           # search window
        "chroma_median_mm": 0.6,        # median on Lab a/b removes colour speckle - applied to PAPER pixels only,
                                        # so thin coloured strokes keep their colour
        "texture_cleanup_pct": 0.5,     # faint dot-like texture above this % of the page -> set those dots to paper
                                        # (only NEUTRAL grey marks no darker than stroke_ink are ever cleared:
                                        #  coloured marks - stamps, seals, ink dots - are never erased)
        "texture_target": 255,          # paper value after Step 6
        "stroke_min_mm2": 0.5,          # ink-loss check counts only connected ink >= this (real strokes, not grain)
    },
    "sharpen": {
        # Decided from sharpness re-measured on the current image; threshold = CONFIG["assess"]["needs"]["sharpen_below"]
        "sigma_mm": 0.2,                # unsharp-mask blur radius (about a stroke width)
        "target_sharpness": 9.0,        # amount = gain x (target - sharpness) / target ...
        "gain": 3.0,
        "min_amount": 0.3, "max_amount": 1.5,   # ... clipped to this range
        "detail_threshold": 4,          # |detail| below this (grey levels) is noise/paper: not boosted
        "halo_overshoot": 8,            # result may exceed the local 3x3 min/max by at most this (no halos)
    },
    "bleed": {
        # Bleed-through = the SAME ink colour appearing as two lightness populations: dark (front) and light
        # (show-through from the back). Ink components are grouped by colour; black/grey is not used (printed text
        # and grey dotted lines always split, even on clean forms).
        "min_component_mm2": 0.3,       # ink components at least this big take part
        "groups": ["blue", "purple", "red"],  # colour groups tested (hue bands from CONFIG["assess"]["ink_hues"])
        "min_components": 100,          # a group needs this many components to judge
        "min_eta": 0.8,                 # Otsu separability (between-class / total variance) of component lightness
        "min_gap_L": 30,                # light mean - dark mean (Lab L, 0-255)
        "light_share": (0.25, 0.85),    # area share of the light population must be in this range (0.15 caught faint
                                        # real handwriting on FIR-4-2013 p5/p18)
        "lighten": 0.6,                 # blend light PIXELS of that colour 60% toward paper: show-through fades but
                                        # anything lightened stays readable if the detection was wrong (was 0.85)
        "protect_mm": 0.25,             # never touch pixels this close to dark same-colour ink (front strokes and their
                                        # soft edges) or to black/grey ink (print): bleed touches both
        "feather_mm": 0.1,
        "paper": 255,
    },
    "contrast": {
        # Decided from ink_level (median grey of strokes) re-measured on the current image;
        # threshold = CONFIG["assess"]["needs"]["faint_ink_level"]
        "target_ink_level": 140,        # gamma chosen so the median ink reaches this grey level
        "knee": 235,                    # lightness >= knee is left unchanged (paper stays white, texture not darkened)
        "max_gamma": 2.5,
    },
    "device_overrides": {},       # any device-profile key, e.g. {"num_workers": 2}
    "review": {                   # thresholds that raise review flags
        "render": {
            "min_short_side_px": 1240,  # ~150 DPI on A4 width
            "min_image_coverage": 0.95, # scan image must cover >= 95% of the page
        },
        "assess": {
            "blank_ink_pct": 0.3,       # below = blank page (later steps skip it)
            "near_blank_ink_pct": 1.0,  # below = nearly blank
            "blurry_below": 5.0,        # sharpness score (synthetic blur sigma 1.5 ~ 4-5)
            "noisy_sigma": 2.0,         # noise sigma
            "specks_per_mp": 500,       # isolated specks / megapixel
            "bleed_faint_stroke_pct": 1.0,
            "large_border_frac": 0.08,  # any border band wider than 8% of that side
            "uneven_illumination": 50,  # paper-level spread (grey levels)
        },
        "borders": {
            "text_at_border_mm2": 20,   # strokes kept inside a band larger than this -> "text_at_border"
            "large_fill_pct": 20,       # filled area above this % of the page -> "large_fill"
            "beyond_spine_ink_mm2": 20, # ink beyond the spine shadow above this -> "content_beyond_spine"
            # "inverted_content_in_border" is always raised when a dark band is left untouched
        },
        "deskew": {
            "large_skew_deg": 5.0,      # rotated by more than this -> "large_skew" (often slanted handwriting)
        },
        "illumination": {
            "ink_loss_pct": 10,         # stroke pixels dropped by more than this % -> "ink_loss"
            "colour_loss_pts": 10,      # blue/purple/red share of strokes DROPPED by more than this -> "colour_loss"
                                        # (a rise is expected: ink on yellow/pink paper regains its true colour)
            "tint_remaining": 4,        # paper chroma after -> "tint_remaining"
            "illumination_remaining": 15,  # paper-level spread after -> "illumination_remaining"
            "gain_capped_pct": 1.0,     # more than this % of the page hit max_gain -> "gain_capped"
            "dark_object_pct": 0.5,     # dark objects protected on more than this % of the page -> "dark_object_kept" (info)
        },
        "denoise": {
            "ink_loss_pct": 8,          # real-stroke pixels (>= stroke_min_mm2) dropped by more than this % -> "ink_loss"
            "noise_remaining_sigma": 2.0,   # noise sigma after -> "noise_remaining"
            "specks_remaining_per_mp": 100, # isolated specks/MP after -> "specks_remaining"
        },
        "sharpen": {
            "still_blurry_below": 5.0,  # sharpness after -> "still_blurry"
            "noise_rise_sigma": 0.5,    # noise sigma rose by more than this -> "noise_amplified"
            "stroke_change_pct": 10,    # real-stroke pixels changed by more than this % -> "ink_change"
        },
        "bleed": {
            "stroke_change_pct": 5,     # dark-population (front) stroke pixels changed by more than this % -> "ink_change"
        },
        "contrast": {
            "still_faint_level": 170,   # median ink grey after -> "still_faint"
            "noise_rise_sigma": 0.5,    # noise sigma rose by more than this -> "noise_amplified"
            "texture_rise_pct": 0.5,    # texture % rose by more than this -> "noise_amplified"
        },
        "output": {
            "bw_ink_recall": 0.85,      # share of stroke CORES (black-hat > stroke_strong, in real strokes) that are black
                                        # in the B&W; below -> "bw_ink_loss" (soft stroke edges are not counted)
            "bw_noise_pct": 1.0,        # black pixels far from any stroke, % of page; above -> "bw_noise"
        },
    },
}

In [ ]:
PROFILE = get_device_profile(CONFIG["device_overrides"])
apply_profile(PROFILE)
print_profile(PROFILE)

In [ ]:
# ---------- Page record helpers (used by every step) ----------

def new_page_record(doc_name: str, doc_path: Path, page_index: int, page, n_pages: int = 0) -> dict:
    w_pt, h_pt = page.get_size()
    return {
        "n_pages": n_pages,
        "doc": doc_name,
        "doc_path": str(doc_path),
        "page_no": page_index + 1,          # 1-based, as people count pages
        "page_index": page_index,           # 0-based, for pypdfium2
        "size_pt": (round(w_pt, 1), round(h_pt, 1)),
        "pdf_orientation": "landscape" if w_pt > h_pt else "portrait",
        "images": {},      # stage name -> np.ndarray (e.g. "original", "clean_color")
        "metrics": {},     # step name -> dict of numbers
        "actions": [],     # (step, what was done)
        "flags": [],       # dicts: step, code, message  -> drives the review filter
        "dpi": None,       # set in Step 1 (estimated or CONFIG override); all physical sizes use it
    }


def add_metrics(record: dict, step: str, **metrics) -> None:
    record["metrics"].setdefault(step, {}).update(metrics)


def add_action(record: dict, step: str, message: str) -> None:
    record["actions"].append((step, message))


def add_flag(record: dict, step: str, code: str, message: str) -> None:
    record["flags"].append({"step": step, "code": code, "message": message})


def mm_to_px(mm: float, dpi: float, odd: bool = False, min_px: int = 1) -> int:
    """Physical size -> pixels at this page's DPI (odd=True for kernel sizes)."""
    px = max(min_px, int(round(mm * dpi / 25.4)))
    return px + 1 if odd and px % 2 == 0 else px


def page_label(record: dict) -> str:
    return f"{record['doc']} p{record['page_no']}"


def run_parallel(fn, records: list[dict], desc: str) -> float:
    """Run fn(record) for every record on PROFILE['num_workers'] threads (OpenCV/NumPy release the GIL).
    Returns elapsed seconds. Errors are not swallowed."""
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=PROFILE["num_workers"]) as pool:
        list(pool.map(fn, records))
    elapsed = time.perf_counter() - t0
    print(f"{desc}: {len(records)} pages in {elapsed:.2f}s ({PROFILE['num_workers']} workers)")
    return elapsed


def show_page_grid(records: list[dict], image_key: str, title: str, cols: int = 8, thumb_h: int = 320) -> None:
    """Thumbnail grid of one image stage; flagged pages get a red title."""
    if not records:
        return
    rows = (len(records) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.0, rows * 2.9), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, rec in zip(axes.flat, records):
        img = rec["images"].get(image_key)
        if img is None:
            continue
        h, w = img.shape[:2]
        thumb = cv2.resize(img, (max(1, int(w * thumb_h / h)), thumb_h), interpolation=cv2.INTER_AREA)
        ax.imshow(cv2.cvtColor(thumb, cv2.COLOR_BGR2RGB) if thumb.ndim == 3 else thumb, cmap="gray")
        flagged = bool(rec["flags"])
        ax.set_title(f"p{rec['page_no']}" + (" ⚑" if flagged else ""), fontsize=9,
                     color="red" if flagged else "black")
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


# ---------- Discover input PDFs and create page records ----------

pdf_dir = Path(CONFIG["input"]["pdf_dir"])
if CONFIG["input"]["pdf_files"]:
    pdf_paths = [pdf_dir / name for name in CONFIG["input"]["pdf_files"]]
else:
    pdf_paths = sorted(pdf_dir.glob("*.pdf"))

missing = [str(p) for p in pdf_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"PDF(s) not found: {missing}")
if not pdf_paths:
    raise FileNotFoundError(f"No PDFs found in {pdf_dir.resolve()}")

OUTPUT_DIR = Path(CONFIG["output"]["dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

page_records: list[dict] = []
print(f"{'Document':<22}{'Pages':>6}{'Selected':>10}{'Portrait':>10}{'Landscape':>11}")
for pdf_path in pdf_paths:
    doc = pdfium.PdfDocument(str(pdf_path))
    n_pages = len(doc)
    first, last = CONFIG["input"]["page_range"] or (1, n_pages)
    selected = range(max(1, first) - 1, min(last, n_pages))
    doc_records = [new_page_record(pdf_path.stem, pdf_path, i, doc[i], n_pages) for i in selected]
    doc.close()
    page_records.extend(doc_records)
    n_land = sum(r["pdf_orientation"] == "landscape" for r in doc_records)
    print(f"{pdf_path.stem:<22}{n_pages:>6}{len(doc_records):>10}{len(doc_records) - n_land:>10}{n_land:>11}")

print(f"\nStep 0 complete: {len(pdf_paths)} document(s), {len(page_records)} page record(s) ready | output -> {OUTPUT_DIR.resolve()}")

## Step 1: Render pages at native scan resolution

**Why:** every sample page is one embedded JPEG scan (~200 DPI). Rendering at 300 DPI only enlarges it, and even rendering at "1×" resamples it and shifts it by 1 px (the image sits at a 1 pt offset).

**Method:**
- If a page is exactly **one image covering the whole page** (no rotation or flips), decode that image directly. This gives the exact scanned pixels, in BGR order ready for OpenCV.
- Otherwise render the page with pdfium at `fallback_dpi` and flag it.
- Optional enlargement (`CONFIG["render"]["upscale"]`, capped by the device profile). Off by default.
- **No filtering.** `images["original"]` is kept untouched for before/after comparison.

**Metrics:** extraction method, pixel size, orientation (from pixels), estimated DPI, coloured-ink share (`color_ink_pct`), paper tint (`paper_tint`), and whether the scan is pure greyscale.

The PDF reports 72 DPI, which is meaningless, so DPI is *estimated* from the short side assuming 8.5-inch-wide paper.

**Review flags:** `low_resolution`, `not_single_scan` (rendered instead of extracted), `page_rotation_set`, `partial_image_coverage`.

In [ ]:
STEP1 = "step1_render"


def find_page_scan(page) -> tuple:
    """Return (image_object, info) if the page is one full-page, axis-aligned scan image, else (None, info)."""
    objects = list(page.get_objects())
    images = [o for o in objects if o.type == pdfium_raw.FPDF_PAGEOBJ_IMAGE]
    w_pt, h_pt = page.get_size()
    info = {"n_objects": len(objects), "n_images": len(images), "pdf_rotation": page.get_rotation(), "coverage": 0.0}
    if len(images) != 1:
        return None, info
    a, b, c, d, _, _ = images[0].get_matrix().get()
    info["coverage"] = round(float(abs(a * d) / (w_pt * h_pt)), 3)
    axis_aligned = b == 0 and c == 0 and a > 0 and d > 0
    if len(objects) == 1 and axis_aligned and info["pdf_rotation"] == 0:
        return images[0], info
    return None, info


def render_page(page, record: dict, cfg: dict, review: dict, max_scale: float) -> np.ndarray:
    t0 = time.perf_counter()
    scan, info = find_page_scan(page)

    if scan is not None:
        record["original_bytes"] = (bytes(scan.get_data(decode_simple=False))  # exact scan, saved as {doc}_{page}_original.jpg
                                    if scan.get_filters() == ["DCTDecode"] else None)
        bitmap = scan.get_bitmap(render=False)          # decoded JPEG pixels, no resampling
        img = np.array(bitmap.to_numpy(), copy=True)    # copy out of pdfium memory (to_numpy is a view)
        if bitmap.format in (pdfium_raw.FPDFBitmap_BGRA, pdfium_raw.FPDFBitmap_BGRx):
            img = img[..., :3]
        elif bitmap.format == pdfium_raw.FPDFBitmap_Gray:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        method = "embedded_image"
        codec = "/".join(scan.get_filters()) or "raw"
        add_action(record, STEP1, f"extracted embedded {codec} scan at native {img.shape[1]}x{img.shape[0]} px")
    else:
        scale = cfg["fallback_dpi"] / 72.0
        bitmap = page.render(scale=scale, rev_byteorder=False)
        img = np.array(bitmap.to_numpy()[..., :3], copy=True)  # BGR(x) -> BGR, copied out of pdfium memory
        bitmap.close()
        method = "rendered"
        codec = None
        record["original_bytes"] = None
        add_action(record, STEP1, f"rendered page at {cfg['fallback_dpi']} DPI (not a single full-page scan)")
        add_flag(record, STEP1, "not_single_scan",
                 f"{info['n_images']} image(s), {info['n_objects']} object(s): rendered instead of extracted")

    img = np.ascontiguousarray(img)
    native_h, native_w = img.shape[:2]

    upscale = min(cfg["upscale"], max_scale)
    if upscale > 1.0:
        img = cv2.resize(img, None, fx=upscale, fy=upscale, interpolation=cv2.INTER_CUBIC)
        add_action(record, STEP1, f"upscaled {upscale}x to {img.shape[1]}x{img.shape[0]} px")

    # Colour content on a 1/4-size copy. Chroma = max - min channel per pixel.
    #   color_ink_pct : % of strongly coloured pixels (blue ink, stamps, tinted paper)
    #   paper_tint    : median chroma of light (paper) pixels; 0 = neutral white/grey paper
    small = cv2.resize(img, None, fx=cfg["color_scale"], fy=cfg["color_scale"], interpolation=cv2.INTER_AREA)
    chroma = small.max(axis=2).astype(np.int16) - small.min(axis=2)
    color_ink_pct = float(np.mean(chroma > cfg["color_chroma_min"]) * 100)
    paper = small.min(axis=2) > cfg["color_paper_min"]
    paper_tint = float(np.median(chroma[paper])) if paper.any() else 0.0
    greyscale_scan = color_ink_pct < cfg["greyscale_max_ink_pct"] and paper_tint < cfg["greyscale_max_tint"]

    short_side = min(native_h, native_w)
    est_dpi = short_side / cfg["assumed_paper_width_in"]
    record["dpi"] = float(cfg["dpi_override"] or est_dpi) * upscale   # may be replaced by the document median below
    add_metrics(
        record, STEP1,
        method=method,
        codec=codec,
        native_px=(native_w, native_h),
        output_px=(img.shape[1], img.shape[0]),
        upscale=upscale,
        orientation="landscape" if native_w > native_h else "portrait",
        est_dpi=round(est_dpi),
        dpi_used=round(record["dpi"]),
        color_ink_pct=round(color_ink_pct, 2),
        paper_tint=round(paper_tint, 1),
        greyscale_scan=greyscale_scan,
        time_ms=round((time.perf_counter() - t0) * 1000, 1),
        **info,
    )

    if short_side < review["min_short_side_px"]:
        add_flag(record, STEP1, "low_resolution",
                 f"short side {short_side}px (~{est_dpi:.0f} DPI) < {review['min_short_side_px']}px")
    if info["pdf_rotation"] != 0:
        add_flag(record, STEP1, "page_rotation_set", f"PDF /Rotate = {info['pdf_rotation']}°")
    if info["n_images"] == 1 and info["coverage"] < review["min_image_coverage"]:
        add_flag(record, STEP1, "partial_image_coverage", f"scan covers {info['coverage']:.0%} of the page")
    return img


# pdfium is not thread-safe and extraction takes ~10 ms/page, so this step runs sequentially.
t_start = time.perf_counter()
for doc_path in dict.fromkeys(r["doc_path"] for r in page_records):
    doc = pdfium.PdfDocument(doc_path)
    for rec in (r for r in page_records if r["doc_path"] == doc_path):
        rec["images"]["original"] = render_page(
            doc[rec["page_index"]], rec, CONFIG["render"], CONFIG["review"]["render"], PROFILE["max_render_scale"]
        )
    doc.close()

# One DPI per document (median of its pages) unless overridden
if CONFIG["render"]["dpi_per_document"] and not CONFIG["render"]["dpi_override"]:
    for doc_name in dict.fromkeys(r["doc"] for r in page_records):
        doc_recs = [r for r in page_records if r["doc"] == doc_name]
        doc_dpi = float(np.median([r["metrics"][STEP1]["est_dpi"] for r in doc_recs]))
        for r in doc_recs:
            r["dpi"] = doc_dpi * r["metrics"][STEP1]["upscale"]
            r["metrics"][STEP1]["dpi_used"] = round(r["dpi"])

elapsed = time.perf_counter() - t_start
total_mb = sum(r["images"]["original"].nbytes for r in page_records) / 1024**2
print(f"Step 1 complete: {len(page_records)} pages in {elapsed:.2f}s | originals in memory: {total_mb:.0f} MB")

In [ ]:
# ---------- Step 1 review: per-page table + thumbnails ----------
print(f"{'Page':<18}{'Method':<16}{'Pixels (WxH)':<14}{'Orient':<11}{'DPI':>5}{'Scan':>8}{'Ink%':>7}{'Tint':>6}{'ms':>7}  Flags")
for rec in page_records:
    m = rec["metrics"][STEP1]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP1) or "-"
    w, h = m["native_px"]
    print(f"{page_label(rec):<18}{m['method']:<16}{f'{w}x{h}':<14}{m['orientation']:<11}{m['dpi_used']:>5}"
          f"{('grey' if m['greyscale_scan'] else 'colour'):>8}{m['color_ink_pct']:>7.2f}{m['paper_tint']:>6.0f}{m['time_ms']:>7}  {flags}")

step1_flagged = [r for r in page_records if any(f["step"] == STEP1 for f in r["flags"])]
print(f"\nFlagged for review in Step 1: {len(step1_flagged)} / {len(page_records)} pages")
for rec in step1_flagged:
    for f in rec["flags"]:
        if f["step"] == STEP1:
            print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")

for doc_name in dict.fromkeys(r["doc"] for r in page_records):
    show_page_grid([r for r in page_records if r["doc"] == doc_name], "original", f"{doc_name}: original (native resolution)")

## Step 2: Page quality assessment (measure only)

Nothing is changed in this step. It measures each page and decides **which later steps the page needs**, so every cleaning step runs only where it helps.

| Metric | How it is measured | Drives |
|---|---|---|
| `borders` | Per side, in the outer 20%: **dark strips** (≥ 3 lines where ≥ 35% of pixels are dark, e.g. a book spine, even when it sits a little inside the edge) and, in the outer 8%, **pale strips** (lines that are evenly off-paper in colour, e.g. a greenish scanner-bed edge). | Step 4 |
| `content_box` | The page minus those border bands. All metrics below are measured inside it. | all |
| `ink_pct` | Stroke pixels (morphological black-hat > 30: locally darker, thinner than 15 px), as % of the content area. | blank detection |
| `sharpness` | High-frequency energy on stroke edges ÷ stroke contrast. Real pages 5–18; synthetic Gaussian blur σ 1.5 ≈ 4–5. | Step 8 |
| `noise_sigma` | Immerkær noise estimate on flat (non-edge) pixels. The scanner clips the paper to white, so clean pages are ≈ 0–1. | Step 7 |
| `texture_pct` | Faint small dots not near any real stroke (toner grain, paper texture). | Step 7 |
| `isolated_specks_per_mp` | Tiny ink blobs (≤ 6 px) with no other ink within 8 px. Dotted fill-in lines don't count. | Step 7 |
| `faint_stroke_pct` | Faint, curvy, stroke-shaped marks not near real ink, after removing ruled/dotted lines. This is the bleed-through signal. | Step 9 |
| `illumination_spread` | 95th − 5th percentile of the estimated paper level (shadows, uneven lighting). | Step 6 |
| `paper_lab`, `paper_tint` | Median Lab colour of paper pixels; chroma distance from neutral grey. | Step 6 |
| `ink_level`, `ink_contrast` | Median grey of strokes; paper − ink. | Step 10 |
| `ink_colors` | Share of strokes that are blue / purple / red / black (by chroma + hue). | Step 6 (keep ink colour) |

**Review flags:** `blank`, `near_blank`, `blurry`, `noisy`, `many_specks`, `bleed_through_suspected`, `large_border`, `uneven_illumination`.

In [ ]:
STEP2 = "step2_assess"
_IMMERKAER = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)


def _border_lines(dark_frac, pale, has_content, max_dark: int, max_pale: int, dark_thr: float,
                  min_dark: int, min_pale: int, grow_frac: float, grow_max: int) -> tuple:
    """Profiles ordered edge -> inward. Returns (lines from the edge that are border, kind).
    Dark strips may sit inside the edge (neighbour-page text outside a spine), so the search skips over content.
    Pale strips must be reached from the edge without crossing any line that has content (text)."""
    dark_inner, run = 0, 0
    for i in range(max_dark):                        # dark strip anywhere in the outer band
        run = run + 1 if dark_frac[i] >= dark_thr else 0
        if run >= min_dark:
            dark_inner = i + 1
    if dark_inner:                                   # shadow edge: grow inward while still partly dark
        grown = 0
        while dark_inner < len(dark_frac) and grown < grow_max and dark_frac[dark_inner] >= grow_frac:
            dark_inner += 1
            grown += 1
    pale_inner, run = 0, 0
    for i in range(max_pale):                        # pale, uniform strip near the edge
        if has_content[i] and not pale[i]:
            break                                    # reached text: the pale band cannot extend past it
        run = run + 1 if pale[i] else 0
        if run >= min_pale:
            pale_inner = i + 1
    kind = {(True, True): "dark+pale", (True, False): "dark", (False, True): "pale"}.get(
        (dark_inner > 0, pale_inner > 0))
    return max(dark_inner, pale_inner), kind


def detect_borders(img: np.ndarray, cfg: dict, dpi: float) -> tuple:
    """Dark (spine/scanner shadow) and pale (scanner bed) bands per side, on a 1/q copy."""
    q = cfg["work_scale"]
    sdpi = dpi / q
    h, w = img.shape[:2]
    small = cv2.resize(img, (w // q, h // q), interpolation=cv2.INTER_AREA)
    sg = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    lab = cv2.cvtColor(small, cv2.COLOR_BGR2LAB).astype(np.float32)
    sh, sw = sg.shape
    center = lab[sh // 4: 3 * sh // 4, sw // 4: 3 * sw // 4].reshape(-1, 3)
    ref = center[center[:, 0] >= np.percentile(center[:, 0], cfg["paper_ref_percentile"])]
    paper_lab = np.median(ref, axis=0)
    paper_grey = float(np.median(cv2.cvtColor(small[sh // 4: 3 * sh // 4, sw // 4: 3 * sw // 4],
                                              cv2.COLOR_BGR2GRAY)))
    de = np.sqrt(((lab - paper_lab) ** 2).sum(axis=2))
    dark = sg < cfg["border_dark_rel"] * paper_grey               # relative to this page's paper
    min_dark = mm_to_px(cfg["border_dark_min_mm"], sdpi)
    min_pale = mm_to_px(cfg["border_pale_min_mm"], sdpi)
    light = lab[..., 0]

    borders = {}
    for side in ("left", "right", "top", "bottom"):
        axis = 0 if side in ("left", "right") else 1        # average down columns / along rows
        dark_frac, mean_de, l_std = dark.mean(axis), de.mean(axis), light.std(axis)
        if side in ("right", "bottom"):
            dark_frac, mean_de, l_std = dark_frac[::-1], mean_de[::-1], l_std[::-1]
        pale = (mean_de >= cfg["border_pale_de"]) & (l_std < cfg["border_pale_std"])
        has_content = l_std >= cfg["border_pale_std"]
        n = len(dark_frac)
        lines, kind = _border_lines(dark_frac, pale, has_content, int(n * cfg["border_dark_search"]),
                                    int(n * cfg["border_pale_search"]), cfg["border_dark_frac"], min_dark, min_pale,
                                    cfg["border_dark_grow_frac"], mm_to_px(cfg["border_dark_grow_max_mm"], sdpi))
        borders[side] = {"px": int(lines * q), "kind": kind}
    return borders, paper_lab


def assess_page(img: np.ndarray, cfg: dict, dpi: float) -> dict:
    t0 = time.perf_counter()
    h, w = img.shape[:2]
    px = lambda mm, odd=False: mm_to_px(mm, dpi, odd)          # physical size -> pixels on this page
    borders, _ = detect_borders(img, cfg, dpi)
    x0, x1 = borders["left"]["px"], w - borders["right"]["px"]
    y0, y1 = borders["top"]["px"], h - borders["bottom"]["px"]
    im = img[y0:y1, x0:x1]
    # Measure at the reference DPI: sharpness, noise and speck size are per-pixel properties
    ref = cfg["reference_dpi"]
    if abs(dpi / ref - 1) > 0.05:
        f = ref / dpi
        im = cv2.resize(im, None, fx=f, fy=f, interpolation=cv2.INTER_AREA if f < 1 else cv2.INTER_CUBIC)
        dpi = ref
        px = lambda mm, odd=False: mm_to_px(mm, dpi, odd)
    g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    area = g.size

    # Strokes: black-hat = how much darker than the local (15 px) background
    sk = px(cfg["stroke_width_max_mm"], odd=True)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (sk, sk))
    bh = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, k)
    weak, ink, strong = bh > cfg["stroke_weak"], bh > cfg["stroke_ink"], bh > cfg["stroke_strong"]
    ink_px = int(ink.sum())

    # Sharpness: high-frequency energy on stroke edges relative to stroke contrast
    bk = px(cfg["sharp_band_mm"], odd=True)
    band = cv2.dilate(ink.astype(np.uint8), np.ones((bk, bk), np.uint8)).astype(bool)
    contrast = float(np.percentile(bh[ink], cfg["contrast_percentile"])) if ink_px else 0.0
    hf = cv2.absdiff(g, cv2.GaussianBlur(g, (0, 0), cfg["sharp_sigma_mm"] * dpi / 25.4))
    sharpness = float(hf[band].mean()) / max(contrast, 1.0) * 100 if ink_px else 0.0

    # Noise: Immerkaer estimate on flat pixels (exclude the 10% strongest gradients)
    grad = cv2.magnitude(cv2.Sobel(g, cv2.CV_32F, 1, 0), cv2.Sobel(g, cv2.CV_32F, 0, 1))
    flat = grad < np.percentile(grad, cfg["noise_flat_percentile"])
    lap = np.abs(cv2.filter2D(g.astype(np.float32), -1, _IMMERKAER))
    noise_sigma = float(np.sqrt(np.pi / 2) * lap[flat].mean() / 6) if flat.any() else 0.0

    # Faint marks not near real strokes, ruled/dotted lines removed -> stroke-shaped (bleed) vs dots (texture)
    wk = weak.astype(np.uint8)
    gap, run_len, pad = px(cfg["rule_line_gap_mm"], odd=True), px(cfg["rule_line_min_mm"], odd=True), px(cfg["rule_line_pad_mm"], odd=True)
    h_lines = cv2.morphologyEx(cv2.dilate(wk, np.ones((1, gap), np.uint8)), cv2.MORPH_OPEN, np.ones((1, run_len), np.uint8))
    v_lines = cv2.morphologyEx(cv2.dilate(wk, np.ones((gap, 1), np.uint8)), cv2.MORPH_OPEN, np.ones((run_len, 1), np.uint8))
    lines = cv2.dilate(h_lines | v_lines, np.ones((pad, pad), np.uint8)).astype(bool)
    nk = px(cfg["near_ink_mm"], odd=True)
    near_strong = cv2.dilate(strong.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (nk, nk))).astype(bool)
    faint = (weak & ~near_strong & ~lines).astype(np.uint8)
    _, _, st, _ = cv2.connectedComponentsWithStats(faint, connectivity=8)
    fa, fw, fh = st[1:, cv2.CC_STAT_AREA], st[1:, cv2.CC_STAT_WIDTH], st[1:, cv2.CC_STAT_HEIGHT]
    mm2_px = (dpi / 25.4) ** 2                                 # pixels per mm^2
    stroke_like = ((fa >= cfg["faint_stroke_min_area_mm2"] * mm2_px)
                   & (np.minimum(fw, fh) >= px(cfg["faint_stroke_min_size_mm"]))
                   & (fa / np.maximum(1, fw * fh) < cfg["faint_stroke_max_fill"]))
    faint_stroke_pct = float(fa[stroke_like].sum()) / area * 100
    texture_pct = float(fa[~stroke_like].sum()) / area * 100

    # Isolated specks: ink blobs <= 6 px with no other ink in a 17x17 window
    _, _, st2, cen = cv2.connectedComponentsWithStats(ink.astype(np.uint8), connectivity=8)
    ia = st2[1:, cv2.CC_STAT_AREA]
    win = px(cfg["speck_isolation_mm"], odd=True)
    box = cv2.boxFilter(ink.astype(np.float32), -1, (win, win), normalize=False)
    cx = np.clip(np.rint(cen[1:, 0]).astype(int), 0, g.shape[1] - 1)
    cy = np.clip(np.rint(cen[1:, 1]).astype(int), 0, g.shape[0] - 1)
    specks = int(((ia <= cfg["speck_max_area_mm2"] * mm2_px) & (box[cy, cx] <= ia + 0.5)).sum())

    # Paper level / illumination on a 1/q copy (closing removes strokes)
    q = cfg["work_scale"]
    sg = cv2.resize(g, (max(1, g.shape[1] // q), max(1, g.shape[0] // q)), interpolation=cv2.INTER_AREA)
    ck = mm_to_px(cfg["illum_close_mm"], dpi / q, odd=True)
    mk = mm_to_px(cfg["illum_median_mm"], dpi / q, odd=True, min_px=3)
    bg = cv2.medianBlur(cv2.morphologyEx(sg, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (ck, ck))), mk)
    lo, hi = cfg["illum_percentiles"]
    illumination_spread = float(np.percentile(bg, hi) - np.percentile(bg, lo))

    # Paper colour (pixels away from any mark) and ink levels/colours
    pk = px(cfg["paper_pad_mm"], odd=True)
    paper = ~cv2.dilate(wk, np.ones((pk, pk), np.uint8)).astype(bool)
    lab = cv2.cvtColor(im, cv2.COLOR_BGR2LAB)
    paper_lab = np.median(lab[paper], axis=0) if paper.any() else np.array([255, 128, 128])
    paper_tint = float(np.hypot(paper_lab[1] - 128.0, paper_lab[2] - 128.0))
    paper_level = float(np.median(g[paper])) if paper.any() else 255.0
    ink_level = float(np.median(g[ink])) if ink_px else paper_level

    chroma = im.max(axis=2).astype(np.int16) - im.min(axis=2)
    hue = cv2.cvtColor(im, cv2.COLOR_BGR2HSV)[..., 0]
    colored = ink & (chroma >= cfg["ink_chroma_min"])
    n_ink = max(1, ink_px)
    def hue_band(lo_hi):                                      # red wraps around 180 -> lo > hi
        lo_h, hi_h = lo_hi
        return (hue >= lo_h) & (hue < hi_h) if lo_h < hi_h else (hue >= lo_h) | (hue < hi_h)
    hues = cfg["ink_hues"]
    blue = int((colored & hue_band(hues["blue"])).sum())
    purple = int((colored & hue_band(hues["purple"])).sum())
    red = int((colored & hue_band(hues["red"])).sum())
    ink_colors = {
        "blue": round(blue / n_ink * 100, 1), "purple": round(purple / n_ink * 100, 1),
        "red": round(red / n_ink * 100, 1), "other_colour": round((int(colored.sum()) - blue - purple - red) / n_ink * 100, 1),
        "black_grey": round((ink_px - int(colored.sum())) / n_ink * 100, 1),
    }

    return {
        "borders": borders,
        "content_box": (int(x0), int(y0), int(x1), int(y1)),
        "content_frac": round((x1 - x0) * (y1 - y0) / (h * w), 3),
        "measured_at_dpi": round(dpi),
        "ink_pct": round(ink_px / area * 100, 2),
        "sharpness": round(sharpness, 2),
        "noise_sigma": round(noise_sigma, 2),
        "texture_pct": round(texture_pct, 2),
        "isolated_specks_per_mp": round(specks / (area / 1e6)),
        "faint_stroke_pct": round(faint_stroke_pct, 2),
        "illumination_spread": round(illumination_spread, 1),
        "paper_level": round(paper_level),
        "paper_lab": [round(float(v), 1) for v in paper_lab],
        "paper_tint": round(paper_tint, 1),
        "ink_level": round(ink_level),
        "ink_contrast": round(paper_level - ink_level),
        "ink_colors": ink_colors,
        "time_ms": round((time.perf_counter() - t0) * 1000),
    }


def decide_needs(rec: dict, m: dict, need: dict, review: dict, ranges: dict) -> dict:
    """Turn metrics into per-step decisions + review flags."""
    h, w = rec["images"]["original"].shape[:2]
    blank = m["ink_pct"] < review["blank_ink_pct"]
    if m["noise_sigma"] >= need["denoise_strong_sigma"]:
        denoise = "strong"
    elif m["noise_sigma"] >= need["denoise_mild_sigma"] or m["texture_pct"] >= need["denoise_texture_pct"]:
        denoise = "mild"
    else:
        denoise = "none"
    needs = {
        "skip_page": blank,
        "border_cleanup": any(b["px"] > 0 for b in m["borders"].values()),
        "illumination": m["illumination_spread"] >= need["illumination_spread"],
        "tint_removal": m["paper_tint"] >= need["paper_tint"],
        "denoise": denoise,
        "speck_removal": m["isolated_specks_per_mp"] >= need["speck_removal_per_mp"],
        "sharpen": (not blank) and m["sharpness"] < need["sharpen_below"],
        "bleed_suppression": m["faint_stroke_pct"] >= need["bleed_faint_stroke_pct"],
        "contrast": (not blank) and m["ink_level"] >= need["faint_ink_level"],
    }

    if blank:
        add_flag(rec, STEP2, "blank", f"ink {m['ink_pct']}% < {review['blank_ink_pct']}%: later steps skip this page")
    elif m["ink_pct"] < review["near_blank_ink_pct"]:
        add_flag(rec, STEP2, "near_blank", f"ink {m['ink_pct']}% < {review['near_blank_ink_pct']}%")
    if not blank and m["sharpness"] < review["blurry_below"]:
        add_flag(rec, STEP2, "blurry", f"sharpness {m['sharpness']} < {review['blurry_below']}")
    if m["noise_sigma"] >= review["noisy_sigma"]:
        add_flag(rec, STEP2, "noisy", f"noise sigma {m['noise_sigma']} >= {review['noisy_sigma']}")
    if m["isolated_specks_per_mp"] >= review["specks_per_mp"]:
        add_flag(rec, STEP2, "many_specks", f"{m['isolated_specks_per_mp']} isolated specks/MP")
    if m["faint_stroke_pct"] >= review["bleed_faint_stroke_pct"]:
        add_flag(rec, STEP2, "bleed_through_suspected", f"faint stroke-like marks {m['faint_stroke_pct']}% of page")
    big = [s for s, b in m["borders"].items()
           if b["px"] >= review["large_border_frac"] * (w if s in ("left", "right") else h)]
    if big:
        add_flag(rec, STEP2, "large_border",
                 ", ".join(f"{s} {m['borders'][s]['px']}px ({m['borders'][s]['kind']})" for s in big))
    if m["illumination_spread"] >= review["uneven_illumination"]:
        add_flag(rec, STEP2, "uneven_illumination", f"paper level spread {m['illumination_spread']}")
    # Unknown-issue guard: anything outside the range seen on the sample set gets a human look
    for key, (lo, hi) in ranges.items():
        if key == "ink_pct" and blank:
            continue
        if not lo <= m[key] <= hi:
            add_flag(rec, STEP2, f"unusual_{key}", f"{key}={m[key]} outside sample range [{lo}, {hi}]: possible new issue")
    return needs


def step2_assess(rec: dict) -> None:
    if rec["metrics"].get(STEP2):              # re-running the cell: start this step clean
        rec["flags"] = [f for f in rec["flags"] if f["step"] != STEP2]
        rec["actions"] = [a for a in rec["actions"] if a[0] != STEP2]
    m = assess_page(rec["images"]["original"], CONFIG["assess"], rec["dpi"])
    rec["metrics"][STEP2] = m
    rec["needs"] = decide_needs(rec, m, CONFIG["assess"]["needs"], CONFIG["review"]["assess"],
                                CONFIG["assess"]["expected_ranges"])
    todo = [k if v is True else f"{k}={v}" for k, v in rec["needs"].items() if v not in (False, "none")]
    add_action(rec, STEP2, "measured only; needs: " + (", ".join(todo) or "nothing"))


run_parallel(step2_assess, page_records, "Step 2 complete")

In [ ]:
# ---------- Step 2 review: metrics table, what each page needs, flagged pages ----------
NEED_SHORT = {"border_cleanup": "B", "illumination": "I", "tint_removal": "T", "speck_removal": "S",
              "sharpen": "Sh", "bleed_suppression": "Bl", "contrast": "C"}


def needs_str(n: dict) -> str:
    if n["skip_page"]:
        return "SKIP (blank)"
    parts = [NEED_SHORT[k] for k in NEED_SHORT if n[k]]
    if n["denoise"] != "none":
        parts.append(f"D:{n['denoise']}")
    return " ".join(parts) or "-"


print(f"{'Page':<16}{'Ink%':>6}{'Sharp':>7}{'Noise':>7}{'Text%':>7}{'Speck':>7}{'Faint%':>8}{'Illum':>7}{'Tint':>6}"
      f"{'InkL':>6}{'Blue%':>7}  {'Borders (px)':<26}{'Needs':<22}Flags")
for rec in page_records:
    m = rec["metrics"][STEP2]
    b = " ".join(f"{s[0].upper()}{v['px']}{ {'dark': 'd', 'pale': 'p', 'dark+pale': 'dp'}[v['kind']] }"
                 for s, v in m["borders"].items() if v["px"]) or "-"
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP2) or "-"
    print(f"{page_label(rec):<16}{m['ink_pct']:>6.1f}{m['sharpness']:>7.1f}{m['noise_sigma']:>7.2f}{m['texture_pct']:>7.2f}"
          f"{m['isolated_specks_per_mp']:>7}{m['faint_stroke_pct']:>8.2f}{m['illumination_spread']:>7.0f}{m['paper_tint']:>6.1f}"
          f"{m['ink_level']:>6}{m['ink_colors']['blue']:>7.1f}  {b:<26}{needs_str(rec['needs']):<22}{flags}")
print("\nBorders: L/R/T/B + px + kind (d = dark strip, p = pale strip, dp = both). "
      "Needs: B border, I illumination, T tint, D denoise, S specks, Sh sharpen, Bl bleed, C contrast.")

# How many pages need each later step
print("\nPages needing each step:")
for key in ["border_cleanup", "illumination", "tint_removal", "speck_removal", "sharpen", "bleed_suppression", "contrast"]:
    print(f"  {key:<18} {sum(bool(r['needs'][key]) for r in page_records):>3} / {len(page_records)}")
for level in ("mild", "strong"):
    print(f"  denoise={level:<10} {sum(r['needs']['denoise'] == level for r in page_records):>3} / {len(page_records)}")

print("\nObserved range in this run vs CONFIG expected_ranges (update ranges only after reviewing outliers):")
for key, (lo, hi) in CONFIG["assess"]["expected_ranges"].items():
    vals = [r["metrics"][STEP2][key] for r in page_records]
    print(f"  {key:<24} observed [{min(vals):>7}, {max(vals):>7}]   expected [{lo}, {hi}]")

step2_flagged = [r for r in page_records if any(f["step"] == STEP2 for f in r["flags"])]
print(f"\nFlagged for review in Step 2: {len(step2_flagged)} / {len(page_records)} pages")
for rec in step2_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP2):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")

In [ ]:
# ---------- Step 2 visual check: detected border bands (red) and content box (green) on flagged pages ----------
def step2_overlay(rec: dict, max_h: int = 700) -> np.ndarray:
    img = rec["images"]["original"]
    h, w = img.shape[:2]
    s = max_h / h
    vis = cv2.resize(img, (int(w * s), max_h), interpolation=cv2.INTER_AREA)
    shade = vis.copy()
    x0, y0, x1, y1 = (int(v * s) for v in rec["metrics"][STEP2]["content_box"])
    mask = np.ones(vis.shape[:2], bool)
    mask[y0:y1, x0:x1] = False
    shade[mask] = (0, 0, 255)
    vis = cv2.addWeighted(shade, 0.35, vis, 0.65, 0)
    cv2.rectangle(vis, (x0, y0), (x1 - 1, y1 - 1), (0, 160, 0), 2)
    return vis


def show_overlays(records: list[dict], title: str, cols: int = 6) -> None:
    if not records:
        print(f"{title}: none")
        return
    rows = (len(records) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 3.8), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, rec in zip(axes.flat, records):
        ax.imshow(cv2.cvtColor(step2_overlay(rec), cv2.COLOR_BGR2RGB))
        codes = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP2)
        ax.set_title(f"{page_label(rec)}\n{codes}", fontsize=8, color="red")
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


show_overlays(step2_flagged, "Step 2 flagged pages: red = detected border band, green = content box")

## Step 3: Orientation by content (0° / 90° / 180° / 270°)

**Rule:** turn a page only when its **text** is sideways or upside down. A landscape page whose text reads normally stays landscape. Only exact 90° turns are used (`cv2.rotate`, lossless); small tilts are Step 5. OpenCV only, no OCR.

**How:**
1. **Text mask** of the content box (Step 2) at 150 DPI: strokes (black-hat), with ruled and dotted lines removed.
2. **Axis (horizontal or vertical text):** close the mask along x and, separately, along y. Horizontal text turns into long flat blobs only when smeared along x.
   - `axis_score` = share of ink in long horizontal blobs − share in long vertical blobs.
   - On the samples, upright pages score ≥ +0.02 and the same pages turned 90° score ≤ −0.02 (all 142 cases correct).
3. **Up/down:** find horizontal bars (Devanagari heading bar शिरोरेखा, T-bars). Upright text has more ink hanging *below* a bar than above it.
   - `updown_score` = vote(page) − vote(page turned 180°).
   - This is reliable for Hindi text but weak for English (the English medical pages score −0.13 to −0.40). So the step is **conservative**: it rotates 180° only at ≤ −0.5 and flags at ≤ −0.15.

**Decisions:**
- Clearly vertical text → turn 90° the way the up/down vote prefers, and flag `rotated`.
- Horizontal text with a strong upside-down vote → turn 180° and flag `rotated`.
- Weak evidence → **no turn**, flag `orientation_uncertain` or `possibly_upside_down` for review.

**Output:** `images["oriented"]`. The Step 2 `content_box` and `borders` are carried over into the new orientation (`metrics["step3_orient"]`).

In [ ]:
STEP3 = "step3_orient"
ROTATIONS = {0: None, 90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}


def rotate(img: np.ndarray, deg_cw: int) -> np.ndarray:
    return img if deg_cw == 0 else cv2.rotate(img, ROTATIONS[deg_cw])


def orientation_text_mask(img: np.ndarray, box: tuple, dpi: float, acfg: dict, work_dpi: float) -> np.ndarray:
    """Binary stroke mask of the content box at work_dpi, ruled/dotted lines removed (same settings as Step 2)."""
    x0, y0, x1, y1 = box
    g = cv2.cvtColor(img[y0:y1, x0:x1], cv2.COLOR_BGR2GRAY)
    f = work_dpi / dpi
    g = cv2.resize(g, None, fx=f, fy=f, interpolation=cv2.INTER_AREA if f < 1 else cv2.INTER_CUBIC)
    px = lambda mm, odd=False: mm_to_px(mm, work_dpi, odd)
    k = px(acfg["stroke_width_max_mm"], odd=True)
    bh = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_RECT, (k, k)))
    ink = (bh > acfg["stroke_ink"]).astype(np.uint8)
    gap, run_len = px(acfg["rule_line_gap_mm"], odd=True), px(acfg["rule_line_min_mm"], odd=True)
    h_lines = cv2.morphologyEx(cv2.dilate(ink, np.ones((1, gap), np.uint8)), cv2.MORPH_OPEN, np.ones((1, run_len), np.uint8))
    v_lines = cv2.morphologyEx(cv2.dilate(ink, np.ones((gap, 1), np.uint8)), cv2.MORPH_OPEN, np.ones((run_len, 1), np.uint8))
    pad = px(acfg["rule_line_pad_mm"], odd=True)
    lines = cv2.dilate(h_lines | v_lines, np.ones((pad, pad), np.uint8))
    return ink & (1 - lines)


def axis_score(mask: np.ndarray, dpi: float, cfg: dict) -> float:
    """> 0: text lines run horizontally; < 0: vertically."""
    k = mm_to_px(cfg["smear_mm"], dpi)
    share = {}
    for name, kernel in (("h", np.ones((1, k), np.uint8)), ("v", np.ones((k, 1), np.uint8))):
        closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        _, _, st, _ = cv2.connectedComponentsWithStats(closed, connectivity=8)
        w, h, a = st[1:, cv2.CC_STAT_WIDTH], st[1:, cv2.CC_STAT_HEIGHT], st[1:, cv2.CC_STAT_AREA]
        long_blob = (w >= cfg["smear_elongation"] * h) if name == "h" else (h >= cfg["smear_elongation"] * w)
        share[name] = float(a[long_blob].sum()) / max(1, int(a.sum()))
    return share["h"] - share["v"]


def bar_vote(mask: np.ndarray, dpi: float, cfg: dict) -> float:
    """(ink below bars - ink above bars) / total, for horizontal bars. > 0 = looks upright."""
    bars = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((1, mm_to_px(cfg["bar_min_mm"], dpi)), np.uint8))
    if not bars.any():
        return 0.0
    d = mm_to_px(cfg["bar_window_mm"], dpi)
    rest = (mask & (1 - cv2.dilate(bars, np.ones((3, 1), np.uint8)))).astype(np.float32)
    k_up = np.zeros((2 * d + 1, 1), np.float32); k_up[:d] = 1          # rows y-d .. y-1
    k_dn = np.zeros((2 * d + 1, 1), np.float32); k_dn[d + 1:] = 1      # rows y+1 .. y+d
    above = cv2.filter2D(rest, -1, k_up, borderType=cv2.BORDER_CONSTANT)[bars.astype(bool)].sum()
    below = cv2.filter2D(rest, -1, k_dn, borderType=cv2.BORDER_CONSTANT)[bars.astype(bool)].sum()
    return float((below - above) / max(1.0, below + above))


def updown_score(mask: np.ndarray, dpi: float, cfg: dict) -> float:
    """vote(as is) - vote(turned 180). > 0 = upright, < 0 = upside down."""
    return bar_vote(mask, dpi, cfg) - bar_vote(cv2.rotate(mask, cv2.ROTATE_180), dpi, cfg)


def detect_orientation(mask: np.ndarray, dpi: float, cfg: dict) -> dict:
    """Return the clockwise turn that makes the text upright, with evidence and a confidence label."""
    if int(mask.sum()) < cfg["min_text_px"]:
        return {"rotation": 0, "decision": "too_little_text", "axis_score": None, "updown_score": None}
    axis = axis_score(mask, dpi, cfg)
    if axis <= -cfg["axis_confident"]:                       # clearly sideways: choose 90 or 270 by up/down vote
        ud_cw = updown_score(cv2.rotate(mask, cv2.ROTATE_90_CLOCKWISE), dpi, cfg)
        rotation = 90 if ud_cw >= 0 else 270
        sure = abs(ud_cw) >= abs(cfg["updown_flag_below"])
        return {"rotation": rotation, "decision": "sideways" if sure else "sideways_direction_unsure",
                "axis_score": round(axis, 3), "updown_score": round(abs(ud_cw), 3)}
    ud = updown_score(mask, dpi, cfg)
    if axis < 0:                                             # leaning vertical but weak: don't turn
        return {"rotation": 0, "decision": "axis_unsure", "axis_score": round(axis, 3), "updown_score": round(ud, 3)}
    if ud <= cfg["updown_rotate_below"]:
        return {"rotation": 180, "decision": "upside_down", "axis_score": round(axis, 3), "updown_score": round(ud, 3)}
    decision = "possibly_upside_down" if ud <= cfg["updown_flag_below"] else "upright"
    return {"rotation": 0, "decision": decision, "axis_score": round(axis, 3), "updown_score": round(ud, 3)}


def rotate_box(box: tuple, w: int, h: int, deg_cw: int) -> tuple:
    """Content box (x0, y0, x1, y1) in a w x h image -> same region after a clockwise turn."""
    x0, y0, x1, y1 = box
    if deg_cw == 90:
        return (h - y1, x0, h - y0, x1)
    if deg_cw == 180:
        return (w - x1, h - y1, w - x0, h - y0)
    if deg_cw == 270:
        return (y0, w - x1, y1, w - x0)
    return box


def rotate_borders(borders: dict, deg_cw: int) -> dict:
    order = ["top", "right", "bottom", "left"]                 # clockwise order of sides
    shift = deg_cw // 90
    return {order[(order.index(side) + shift) % 4]: v for side, v in borders.items()}


def step3_orient(rec: dict) -> None:
    if rec["metrics"].get(STEP3):                              # re-running the cell: start this step clean
        rec["flags"] = [f for f in rec["flags"] if f["step"] != STEP3]
        rec["actions"] = [a for a in rec["actions"] if a[0] != STEP3]
    cfg, img = CONFIG["orient"], rec["images"]["original"]
    s2 = rec["metrics"][STEP2]
    h, w = img.shape[:2]

    if rec["needs"]["skip_page"]:
        result = {"rotation": 0, "decision": "blank_not_checked", "axis_score": None, "updown_score": None}
    else:
        mask = orientation_text_mask(img, s2["content_box"], rec["dpi"], CONFIG["assess"], cfg["work_dpi"])
        result = detect_orientation(mask, cfg["work_dpi"], cfg)

    deg = result["rotation"]
    rec["images"]["oriented"] = rotate(img, deg)              # no copy when deg == 0
    oh, ow = rec["images"]["oriented"].shape[:2]
    rec["metrics"][STEP3] = {
        **result,
        "content_box": rotate_box(s2["content_box"], w, h, deg),
        "borders": rotate_borders(s2["borders"], deg),
        "orientation_before": "landscape" if w > h else "portrait",
        "orientation_after": "landscape" if ow > oh else "portrait",
    }

    if deg:
        add_action(rec, STEP3, f"turned {deg}° clockwise ({result['decision']}, axis {result['axis_score']}, "
                               f"up/down {result['updown_score']})")
        add_flag(rec, STEP3, "rotated", f"turned {deg}° clockwise: confirm the page now reads upright")
    else:
        add_action(rec, STEP3, f"kept as is ({result['decision']})")
    if result["decision"] == "sideways_direction_unsure":
        add_flag(rec, STEP3, "orientation_uncertain", "text is sideways but 90° vs 270° is unsure: may be upside down now")
    elif result["decision"] == "axis_unsure":
        add_flag(rec, STEP3, "orientation_uncertain", f"text direction unclear (axis score {result['axis_score']})")
    elif result["decision"] == "possibly_upside_down":
        add_flag(rec, STEP3, "possibly_upside_down",
                 f"weak upside-down evidence (up/down {result['updown_score']}): not turned")
    elif result["decision"] == "too_little_text":
        add_flag(rec, STEP3, "orientation_unchecked", "too little text to judge orientation")


run_parallel(step3_orient, page_records, "Step 3 complete")

In [ ]:
# ---------- Step 3 review: decisions, flagged pages, optional self-test ----------
print(f"{'Page':<16}{'Turn':>6}  {'Decision':<26}{'Axis':>7}{'Up/down':>9}  {'Before -> after':<22}Flags")
for rec in page_records:
    m = rec["metrics"][STEP3]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP3) or "-"
    fmt = lambda v: "-" if v is None else f"{v:+.3f}"
    print(f"{page_label(rec):<16}{m['rotation']:>5}°  {m['decision']:<26}{fmt(m['axis_score']):>7}{fmt(m['updown_score']):>9}  "
          f"{m['orientation_before'] + ' -> ' + m['orientation_after']:<22}{flags}")

turned = [r for r in page_records if r["metrics"][STEP3]["rotation"]]
step3_flagged = [r for r in page_records if any(f["step"] == STEP3 for f in r["flags"])]
print(f"\nTurned: {len(turned)} | flagged for review: {len(step3_flagged)} / {len(page_records)}")
for rec in step3_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP3):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")

if step3_flagged:
    show_page_grid(step3_flagged, "oriented", "Step 3 flagged pages (after orientation)", cols=6)

# Self-test: turn real pages by a known angle and check the detector undoes it (slow: 3 extra passes per page)
RUN_ORIENTATION_SELF_TEST = False
if RUN_ORIENTATION_SELF_TEST:
    cfg = CONFIG["orient"]
    results = {"correct": 0, "not_turned": 0, "wrong_turn": 0}
    for rec in [r for r in page_records if not r["needs"]["skip_page"]]:
        base = orientation_text_mask(rec["images"]["original"], rec["metrics"][STEP2]["content_box"],
                                     rec["dpi"], CONFIG["assess"], cfg["work_dpi"])
        for applied in (90, 180, 270):
            got = detect_orientation(rotate(base, applied), cfg["work_dpi"], cfg)["rotation"]
            key = "correct" if (got + applied) % 360 == 0 else ("not_turned" if got == 0 else "wrong_turn")
            results[key] += 1
    print("Orientation self-test on turned copies:", results)

## Step 4: Border & scanner-artifact cleanup

**Why:** black book-spine and scanner shadows, slivers of the neighbouring page, and pale scanner-bed strips throw off the thresholds, deskew and background estimate in later steps.

**Input:** `images["oriented"]` and the border bands carried through Step 3. Only pages with `needs["border_cleanup"]` are changed.

**Method:**
1. **Fill, don't crop:** pale bands (scanner bed, off-page area) are replaced with the page's **paper colour** (Step 2 `paper_lab`), so the page keeps its size and Step 6 later whitens it together with the rest of the paper.
   **Dark bands (spine shadow) are flattened, not painted:** page text is often printed *on* the shadow (row numbers, the start of field labels). The local background (grey closing wider than any stroke) is divided out, so the shadow turns to paper while the text stays. Only background too dark to hold any text (below paper ÷ `shadow_max_gain`) is filled. The area beyond the spine is flattened too, **never erased**: on these samples it is sometimes a neighbour-page sliver and sometimes this page's own lines continuing across the spine, which pixels alone cannot tell apart. If it holds ink, the page is flagged `content_beyond_spine`.
2. **Protect text at the edge:** ink strokes (black-hat) inside the content box within `protect_mm` of a band are followed into the band. Any stroke that crosses the boundary (a descender, the last line on a short sheet) is restored from the original. Neighbour-page text outside a spine doesn't touch the content box, so it is still removed.
3. **Feather:** the fill blends in over `feather_mm` so later filters don't see a hard step.
4. **Verify:** Step 2's border detector runs again on the result. Any border still found is flagged.

**Review flags:** `text_at_border` (strokes crossed into a band and were kept), `content_beyond_spine`, `large_fill` (> 20% of the page filled), `border_remaining`.

**Safety (review): nothing inside a dark band is removed.** Both sides of the spine can hold text: this page's row numbers and field labels (inside the shadow) and the neighbour page's sliver (beyond it).
- Recoverable shadow → **flattened** (text stays), which is the reviewed default. An optional **monotone tone lift** inside the band (`shadow_lift: (0.45, 0.80)`, off by default) keeps lightness below 45% of paper (text) unchanged, turns lightness above 80% (shadow grain) into paper, and blends smoothly in between. Because the curve never reorders tones, no stroke can disappear; (0.40, 0.70) was judged too strong. *Rejected:* binary "text or not" masks (local-median background, hysteresis on the flattened band). They erased "9" of "9654", "(i)–(iv)", "Type of" and "Place of" on FIR-22-2010 p1.
- Too dark to flatten (darker than paper ÷ 6) → **left as the original pixels** (`fill_unrecoverable_dark: False`). Painting it with paper is cosmetic only and could hide white-on-black content, so it is off.
- Bright glyph-shaped marks on a solid dark background (≥ 60% black within 4 mm) are reported as `inverted_content_in_border` for review.
- Only **pale** bands (scanner bed) are ever painted, and pen strokes crossing into them are restored.

In [ ]:
STEP4 = "step4_borders"


def paper_bgr(paper_lab) -> np.ndarray:
    lab = np.uint8([[paper_lab]])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)[0, 0].astype(np.float32)


def dark_band_split(g: np.ndarray, borders: dict, dpi: float, smooth_mm: float, edge_rel: float) -> tuple:
    """For each dark side, find where the shadow starts on the edge side of its darkest line.
    Returns (outer mask: edge..shadow start = neighbour page / beyond the spine, to fill;
             inner mask: shadow start..band edge, to flatten so text printed on the shadow survives)."""
    h, w = g.shape
    outer, inner = np.zeros((h, w), bool), np.zeros((h, w), bool)
    sig = max(1.0, smooth_mm * dpi / 25.4)
    for side, b in borders.items():
        if not b["px"] or "dark" not in (b["kind"] or ""):
            continue
        n = b["px"]
        if side in ("left", "right"):
            prof = (g[:, :n] if side == "left" else g[:, w - n:][:, ::-1]).mean(axis=0)
        else:
            prof = (g[:n, :] if side == "top" else g[h - n:, :][::-1, :]).mean(axis=1)
        prof = cv2.GaussianBlur(prof.astype(np.float32).reshape(1, -1), (0, 0), sig).ravel()
        darkest = int(np.argmin(prof))
        paper_lvl = float(np.percentile(g, 90))
        level = prof[darkest] + edge_rel * (paper_lvl - prof[darkest])
        crease = darkest                                       # walk from the darkest line toward the edge
        while crease > 0 and prof[crease - 1] <= level:
            crease -= 1                                        # lines [0, crease) are beyond the shadow -> fill
        o, i = np.zeros((h, w), bool), np.zeros((h, w), bool)
        if side == "left":
            o[:, :crease] = True; i[:, crease:n] = True
        elif side == "right":
            o[:, w - crease:] = True; i[:, w - n:w - crease] = True
        elif side == "top":
            o[:crease, :] = True; i[crease:n, :] = True
        else:
            o[h - crease:, :] = True; i[h - n:h - crease, :] = True
        outer |= o
        inner |= i
    return outer, inner & ~outer


def dark_band_mask(h: int, w: int, borders: dict) -> np.ndarray:
    """Union of the dark (spine/shadow) border bands, from the edges."""
    dark = np.zeros((h, w), bool)
    for side, b in borders.items():
        if not b["px"] or "dark" not in (b["kind"] or ""):
            continue
        n = b["px"]
        if side == "left":
            dark[:, :n] = True
        elif side == "right":
            dark[:, w - n:] = True
        elif side == "top":
            dark[:n, :] = True
        else:
            dark[h - n:, :] = True
    return dark


def shadow_lift(img: np.ndarray, region: np.ndarray, paper_bgr_v: np.ndarray, dpi: float, lift: tuple,
                feather_mm: float) -> np.ndarray:
    """Monotone curve on Lab L inside region: L <= a*paper unchanged, L >= b*paper -> paper, smoothstep between."""
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    paper_l = float(cv2.cvtColor(paper_bgr_v.reshape(1, 1, 3).astype(np.uint8), cv2.COLOR_BGR2LAB)[0, 0, 0])
    a, b = lift[0] * paper_l, lift[1] * paper_l
    x = np.arange(256, dtype=np.float32)
    t = np.clip((x - a) / max(1.0, b - a), 0, 1)
    t = t * t * (3 - 2 * t)
    lut = np.clip(x + t * (paper_l - x) + 0.5, 0, 255).astype(np.uint8)
    w = cv2.GaussianBlur(region.astype(np.float32), (0, 0), max(1.0, feather_mm * dpi / 25.4))
    L = lab[..., 0].astype(np.float32)
    lab[..., 0] = np.clip(L * (1 - w) + cv2.LUT(lab[..., 0], lut).astype(np.float32) * w + 0.5, 0, 255).astype(np.uint8)
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def inverted_glyphs(g: np.ndarray, region: np.ndarray, dpi: float, cfg: dict, acfg: dict, paper_level: float) -> tuple:
    """Bright, glyph-shaped marks on a dark background inside region (white-on-black print). Returns (count, mm2)."""
    if not region.any():
        return 0, 0.0
    sk = mm_to_px(acfg["stroke_width_max_mm"], dpi, odd=True)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (sk, sk))
    tophat = cv2.morphologyEx(g, cv2.MORPH_TOPHAT, k)
    black = (g < cfg["inverted_bg_rel"] * paper_level).astype(np.float32)
    win = mm_to_px(cfg["inverted_solid_mm"], dpi, odd=True)
    dark_bg = cv2.boxFilter(black, -1, (win, win)) >= cfg["inverted_solid_frac"]   # solid dark surroundings
    bright = ((tophat > cfg["inverted_contrast_rel"] * paper_level) & dark_bg & region).astype(np.uint8)
    _, _, st, _ = cv2.connectedComponentsWithStats(bright, connectivity=8)
    mm = 25.4 / dpi
    w, h, a = st[1:, cv2.CC_STAT_WIDTH] * mm, st[1:, cv2.CC_STAT_HEIGHT] * mm, st[1:, cv2.CC_STAT_AREA] * mm * mm
    lo, hi = cfg["inverted_glyph_size_mm"]
    short, long_ = np.minimum(w, h), np.maximum(w, h)
    glyph = ((a >= cfg["inverted_min_glyph_mm2"]) & (short >= lo) & (long_ <= hi)
             & (long_ / np.maximum(short, 1e-6) <= cfg["inverted_max_elongation"]))
    return int(glyph.sum()), float(a[glyph].sum())


def clean_borders(img: np.ndarray, box: tuple, borders: dict, fill_bgr: np.ndarray, dpi: float,
                  cfg: dict, acfg: dict) -> tuple:
    """Pale bands: fill with fill_bgr, keeping strokes that cross into them.
    Dark bands: flatten the shadow (keeps text printed on it); fill only unrecoverable / detached parts.
    Returns (cleaned image, filled mask, restored-stroke mask, flattened mask)."""
    h, w = img.shape[:2]
    x0, y0, x1, y1 = box
    px = lambda mm, odd=False: mm_to_px(mm, dpi, odd)

    band = np.ones((h, w), bool)
    band[y0:y1, x0:x1] = False
    if not band.any():
        return img, band, np.zeros_like(band), np.zeros_like(band), np.zeros_like(band), {
            "inverted_glyphs": 0, "inverted_mm2": 0.0, "inverted_found": False, "dark_untouched_pct": 0.0}

    # Strokes on the whole page, then the ones that cross the content-box edge
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sk = px(acfg["stroke_width_max_mm"], odd=True)
    bh = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_RECT, (sk, sk)))
    ink = (bh > acfg["stroke_ink"]).astype(np.uint8)
    _, labels, stats, _ = cv2.connectedComponentsWithStats(ink, connectivity=8)
    p = px(cfg["protect_mm"])
    inner_ring = np.zeros((h, w), bool)
    inner_ring[max(0, y0 - 1):min(h, y1 + 1), max(0, x0 - 1):min(w, x1 + 1)] = True
    inner_ring[y0 + p:max(y0 + p, y1 - p), x0 + p:max(x0 + p, x1 - p)] = False
    inner_ring &= ~band
    crossing = np.intersect1d(np.unique(labels[inner_ring & (ink > 0)]), np.unique(labels[band & (ink > 0)]))
    crossing = crossing[crossing > 0]
    max_area = cfg["restore_max_stroke_mm2"] * (dpi / 25.4) ** 2
    crossing = crossing[stats[crossing, cv2.CC_STAT_AREA] <= max_area]        # pen strokes only, not shadow texture
    if crossing.size:                                                         # ... and thin: widest point of each blob
        width = cv2.distanceTransform(ink, cv2.DIST_L2, 3) * 2
        widest = np.zeros(labels.max() + 1, np.float32)
        np.maximum.at(widest, labels[ink > 0], width[ink > 0])
        crossing = crossing[widest[crossing] <= px(cfg["restore_max_stroke_width_mm"])]
    content = (~band).astype(np.uint8)
    reach = cv2.distanceTransform(1 - content, cv2.DIST_L2, 3) <= px(cfg["restore_reach_mm"])
    local_bg = cv2.morphologyEx(g, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (sk, sk)))
    paper_level = float(np.median(g[~band])) if (~band).any() else 255.0
    on_paper = local_bg >= cfg["restore_min_background_rel"] * paper_level
    restored = np.isin(labels, crossing) & band & reach & on_paper
    if restored.any():
        rk = px(cfg["restore_pad_mm"], odd=True)
        restored = cv2.dilate(restored.astype(np.uint8), np.ones((rk, rk), np.uint8)).astype(bool) & band

    # Dark bands: divide out the shadow (keeps text printed on it). The part beyond the spine ("outer") is
    # flattened too, never erased: it may be this page continuing across the spine. Its ink is reported.
    work = img.astype(np.float32)
    outer, inner = dark_band_split(g, borders, dpi, cfg["crease_smooth_mm"], cfg["shadow_edge_rel"])
    outer &= band
    inner = (inner | outer) & band
    beyond_ink = outer & (ink > 0)
    flattened = np.zeros((h, w), bool)
    # White-on-dark marks inside the dark band (e.g. a black title banner near the edge) are reported for review.
    # They are safe either way: flattening divides by the local BRIGHT level (gain ~1 on a banner), and parts too
    # dark to flatten are left as original (never painted) unless fill_unrecoverable_dark is set.
    n_glyphs, glyph_mm2 = inverted_glyphs(g, inner, dpi, cfg, acfg, paper_level)
    dark_untouched = np.zeros((h, w), bool)
    if inner.any():
        sm = max(1.0, cfg["shadow_smooth_mm"] * dpi / 25.4)
        bk = px(cfg["shadow_bg_mm"], odd=True)
        bg = cv2.morphologyEx(cv2.GaussianBlur(g, (0, 0), sm), cv2.MORPH_CLOSE,
                              cv2.getStructuringElement(cv2.MORPH_RECT, (bk, bk)))
        bg = cv2.GaussianBlur(bg, (0, 0), bk / 2).astype(np.float32)
        recoverable = (bg >= paper_level / cfg["shadow_max_gain"]).astype(np.uint8)
        mk = px(cfg["shadow_mask_smooth_mm"], odd=True)
        recoverable = cv2.morphologyEx(cv2.morphologyEx(recoverable, cv2.MORPH_OPEN, np.ones((mk, mk), np.uint8)),
                                       cv2.MORPH_CLOSE, np.ones((mk, mk), np.uint8)).astype(bool)
        flattened = inner & recoverable
        if not cfg["fill_unrecoverable_dark"]:
            dark_untouched = inner & ~recoverable              # too dark to flatten: keep the original pixels
        # Flatten lightness only (Lab L x gain); colour channels untouched so JPEG chroma noise is not amplified
        gain = np.where(flattened, paper_level / np.maximum(bg, 1.0), 1.0)
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB).astype(np.float32)
        paper_l = float(cv2.cvtColor(fill_bgr.reshape(1, 1, 3).astype(np.uint8), cv2.COLOR_BGR2LAB)[0, 0, 0])
        lab[..., 0] = np.minimum(lab[..., 0] * gain, paper_l)
        flat = cv2.cvtColor(np.clip(lab + 0.5, 0, 255).astype(np.uint8), cv2.COLOR_LAB2BGR).astype(np.float32)
        # Amplified scanner grain: where the blurred result is close to paper there is no stroke -> paper colour
        fg = cv2.GaussianBlur(cv2.cvtColor(flat.astype(np.uint8), cv2.COLOR_BGR2GRAY), (0, 0),
                              max(0.5, cfg["shadow_grain_blur_mm"] * dpi / 25.4))
        grain = flattened & (fg >= cfg["shadow_grain_rel"] * paper_level)
        flat[grain] = fill_bgr
        work = np.where(flattened[..., None], flat, work)

    if cfg["shadow_lift"] and flattened.any():                 # lighten the flattened shadow grain, keep text
        region = dark_band_mask(h, w, borders) & band
        work = shadow_lift(np.clip(work + 0.5, 0, 255).astype(np.uint8), region, fill_bgr, dpi,
                           cfg["shadow_lift"], cfg["shadow_lift_feather_mm"]).astype(np.float32)

    fill = band & ~restored & ~flattened & ~dark_untouched
    # Feather: alpha rises from 0 at the kept region to 1 at feather_mm inside the filled area
    keep = (~fill).astype(np.uint8)
    dist = cv2.distanceTransform(1 - keep, cv2.DIST_L2, 3)
    alpha = np.clip(dist / max(1.0, px(cfg["feather_mm"])), 0, 1)[..., None]
    out = work * (1 - alpha) + fill_bgr[None, None, :] * alpha
    info = {"inverted_glyphs": n_glyphs, "inverted_mm2": glyph_mm2,
            "inverted_found": n_glyphs >= cfg["inverted_min_glyphs"],
            "dark_untouched_pct": round(float(dark_untouched.mean() * 100), 2)}
    return np.clip(out + 0.5, 0, 255).astype(np.uint8), fill, restored, flattened, beyond_ink, info


def step4_borders(rec: dict) -> None:
    if rec["metrics"].get(STEP4):                              # re-running the cell: start this step clean
        rec["flags"] = [f for f in rec["flags"] if f["step"] != STEP4]
        rec["actions"] = [a for a in rec["actions"] if a[0] != STEP4]
    cfg, review = CONFIG["borders"], CONFIG["review"]["borders"]
    img = rec["images"]["oriented"]
    s3 = rec["metrics"][STEP3]
    if not rec["needs"]["border_cleanup"] or rec["needs"]["skip_page"]:
        rec["images"]["borders_clean"] = img                   # unchanged (no copy)
        rec["metrics"][STEP4] = {"applied": False, "filled_pct": 0.0, "restored_mm2": 0.0}
        add_action(rec, STEP4, "no border to clean")
        return

    t0 = time.perf_counter()
    fill_bgr = (np.array([255, 255, 255], np.float32) if cfg["fill"] == "white"
                else paper_bgr(rec["metrics"][STEP2]["paper_lab"]))
    out, filled, restored, flattened, beyond_ink, inv = clean_borders(img, s3["content_box"], s3["borders"], fill_bgr, rec["dpi"],
                                                     cfg, CONFIG["assess"])
    rec["images"]["borders_clean"] = out

    h, w = out.shape[:2]
    mm2 = (25.4 / rec["dpi"]) ** 2
    filled_pct = float(filled.mean() * 100)
    restored_mm2 = float(restored.sum() * mm2)
    beyond_mm2 = float(beyond_ink.sum() * mm2)
    n_glyphs, glyph_mm2, inv_found = inv["inverted_glyphs"], inv["inverted_mm2"], inv["inverted_found"]
    residual, _ = detect_borders(out, CONFIG["assess"], rec["dpi"])     # verify: no dark strip should be left
    remaining = {s: b["px"] for s, b in residual.items() if b["px"] > 0 and "dark" in (b["kind"] or "")}
    pale_left = {s: b["px"] for s, b in residual.items() if b["kind"] == "pale"}
    rec["metrics"][STEP4] = {
        "applied": True,
        "fill_bgr": [round(float(v)) for v in fill_bgr],
        "filled_pct": round(filled_pct, 2),
        "shadow_flattened_pct": round(float(flattened.mean() * 100), 2),
        "restored_mm2": round(restored_mm2, 1),
        "beyond_spine_ink_mm2": round(beyond_mm2, 1),
        "inverted_glyphs": n_glyphs, "dark_kept_original_pct": inv["dark_untouched_pct"],
        "sides": {s: b["px"] for s, b in s3["borders"].items() if b["px"]},
        "remaining_after": remaining,
        "pale_after": pale_left,              # info only: filled bands can differ slightly from the paper reference
        "time_ms": round((time.perf_counter() - t0) * 1000),
    }
    sides = ", ".join(f"{s} {v}px" for s, v in rec["metrics"][STEP4]["sides"].items())
    add_action(rec, STEP4, f"filled {filled_pct:.1f}% of page with paper colour ({sides}); "
                           f"flattened {flattened.mean() * 100:.1f}% (spine shadow, text kept); "
                           f"kept {restored_mm2:.1f} mm² of strokes crossing the edge")
    if restored_mm2 > review["text_at_border_mm2"]:
        add_flag(rec, STEP4, "text_at_border", f"{restored_mm2:.1f} mm² of strokes cross into the border band and were kept: "
                                               "check the band did not cut content")
    if inv_found:
        add_flag(rec, STEP4, "inverted_content_in_border",
                 f"{n_glyphs} white-on-dark glyph-like marks ({glyph_mm2:.0f} mm²) inside a dark border band: "
                 "check them (flattened band keeps them; too-dark parts kept as original)")
    if beyond_mm2 > review["beyond_spine_ink_mm2"]:
        add_flag(rec, STEP4, "content_beyond_spine",
                 f"{beyond_mm2:.0f} mm² of ink beyond the spine shadow was kept: neighbour-page sliver or this page "
                 "continuing across the spine? (not erased)")
    if filled_pct > review["large_fill_pct"]:
        add_flag(rec, STEP4, "large_fill", f"{filled_pct:.1f}% of the page was filled")
    if remaining:
        add_flag(rec, STEP4, "border_remaining", "dark border still detected after cleanup: " +
                 ", ".join(f"{s} {v}px" for s, v in remaining.items()))


run_parallel(step4_borders, page_records, "Step 4 complete")

In [ ]:
# ---------- Step 4 review: what was filled, flags, before/after ----------
changed = [r for r in page_records if r["metrics"][STEP4]["applied"]]
print(f"{'Page':<16}{'Filled %':>9}{'Flat %':>8}{'Kept mm²':>10}  {'Sides (px)':<36}{'Remaining':<16}Flags")
for rec in changed:
    m = rec["metrics"][STEP4]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP4) or "-"
    sides = " ".join(f"{s[0].upper()}{v}" for s, v in m["sides"].items())
    rem = " ".join(f"{s[0].upper()}{v}" for s, v in m["remaining_after"].items()) or "-"
    print(f"{page_label(rec):<16}{m['filled_pct']:>9.1f}{m['shadow_flattened_pct']:>8.1f}{m['restored_mm2']:>10.1f}  "
          f"{sides:<36}{rem:<16}{flags}")
print(f"\nCleaned {len(changed)} / {len(page_records)} pages")

step4_flagged = [r for r in page_records if any(f["step"] == STEP4 for f in r["flags"])]
print(f"Flagged for review in Step 4: {len(step4_flagged)}")
for rec in step4_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP4):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")


def show_before_after(records: list[dict], before: str, after: str, title: str, step: str,
                      max_pages: int = 12, h: int = 520) -> None:
    """Side-by-side before|after thumbnails; titles show that step's flags."""
    records = records[:max_pages]
    if not records:
        return
    cols = 3
    rows = (len(records) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 5.2, rows * 4.2), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, rec in zip(axes.flat, records):
        pair = []
        for key in (before, after):
            im = rec["images"][key]
            im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR) if im.ndim == 2 else im      # B&W pages
            s = h / im.shape[0]
            pair.append(cv2.resize(im, (int(im.shape[1] * s), h), interpolation=cv2.INTER_AREA))
        sep = np.full((h, 8, 3), 255, np.uint8)
        ax.imshow(cv2.cvtColor(np.hstack([pair[0], sep, pair[1]]), cv2.COLOR_BGR2RGB))
        codes = ", ".join(f["code"] for f in rec["flags"] if f["step"] == step)
        ax.set_title(f"{page_label(rec)}  (before | after)" + (f"\n⚑ {codes}" if codes else ""), fontsize=9,
                     color="red" if codes else "black")
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


# Largest changes first, flagged pages always included
biggest = sorted(changed, key=lambda r: -r["metrics"][STEP4]["filled_pct"])
show_list = step4_flagged + [r for r in biggest if r not in step4_flagged]
show_before_after(show_list, "oriented", "borders_clean", "Step 4: border cleanup (flagged + largest fills)", STEP4)

## Step 5: Deskew (small angle, ±10°)

**Why:** most scans are tilted by a fraction of a degree to a few degrees. Straight text lines help reading, later steps and any future OCR.

**Input:** `images["borders_clean"]` and the content box (Step 3). Measured on a 150 DPI copy of the content box only, so borders don't take part.

**Two independent estimates:**
1. **Projection (text):** the stroke mask with ruled/dotted lines removed is rotated through ±10°, coarse then fine (0.05°). The angle where text rows are sharpest (largest sum of squared row-to-row profile changes) wins. Works on handwritten pages.
2. **Printed lines (forms):** Hough segments ≥ 30 mm long (form rules, dotted fill-in lines, gaps ≤ 1.5 mm bridged), within ±10° of horizontal. Their angles are combined as a length-weighted median.

**Decision (thresholds measured with the self-test on tilted sample pages):**
- **Strong lines** (≥ 10 segments, ≥ 1 m total) → line angle. Median error 0.09°, 95% within 0.5°. This also works on handwriting, because Hough follows the long Devanagari heading bars.
- Otherwise **strong projection** (text rows ≥ 1.1× sharper at the best angle than at 0°) → projection angle. If there are some (weak) lines that differ from it by more than 1°, the page is treated as uncertain instead.
- Otherwise both weak but agreeing within 0.5° → line angle.
- Otherwise → **not rotated**, flagged `deskew_uncertain` if either estimate suggests ≥ 0.5°.
- Under 0.1° → leave the page alone.
- If the writing itself is slanted (lines rising 5–6° on a straight sheet), the step levels the **text**, because that is what matters for reading and extraction, and flags it `large_skew`.

**Rotation:** around the centre with an **expanded canvas** (no corner is clipped). New area gets the paper colour. Cubic interpolation, on the device's warp backend (OpenCL on the Mac, CUDA on GB10 when available). Portrait/landscape never changes.

**Review flags:** `deskew_disagree` (both estimates strong but different), `large_skew` (> 5°), `deskew_uncertain`.

In [ ]:
STEP5 = "step5_deskew"


def deskew_masks(img: np.ndarray, box: tuple, dpi: float, acfg: dict, work_dpi: float) -> tuple:
    """(text strokes without ruled lines, all strokes) of the content box at work_dpi."""
    x0, y0, x1, y1 = box
    g = cv2.cvtColor(img[y0:y1, x0:x1], cv2.COLOR_BGR2GRAY)
    f = work_dpi / dpi
    g = cv2.resize(g, None, fx=f, fy=f, interpolation=cv2.INTER_AREA if f < 1 else cv2.INTER_CUBIC)
    px = lambda mm, odd=False: mm_to_px(mm, work_dpi, odd)
    k = px(acfg["stroke_width_max_mm"], odd=True)
    bh = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_RECT, (k, k)))
    ink = (bh > acfg["stroke_ink"]).astype(np.uint8)
    gap, run_len = px(acfg["rule_line_gap_mm"], odd=True), px(acfg["rule_line_min_mm"], odd=True)
    h_lines = cv2.morphologyEx(cv2.dilate(ink, np.ones((1, gap), np.uint8)), cv2.MORPH_OPEN, np.ones((1, run_len), np.uint8))
    v_lines = cv2.morphologyEx(cv2.dilate(ink, np.ones((gap, 1), np.uint8)), cv2.MORPH_OPEN, np.ones((run_len, 1), np.uint8))
    lines = cv2.dilate(h_lines | v_lines, np.ones((3, 3), np.uint8))
    return ink & (1 - lines), ink


def _profile_score(mask: np.ndarray, angle: float) -> float:
    """Sharpness of the row profile after rotating by angle. Nearest-neighbour on purpose: interpolation would
    blur every angle except 0° and bias the search towards 'no skew'."""
    h, w = mask.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rot = cv2.warpAffine(mask, M, (w, h), flags=cv2.INTER_NEAREST, borderValue=0)
    p = rot.sum(axis=1, dtype=np.float64)
    return float(np.sum(np.diff(p) ** 2))


def projection_skew(text_mask: np.ndarray, cfg: dict) -> tuple:
    """Correction angle (deg, cv2 convention: rotate by this to straighten) and peak sharpness vs 0°."""
    if int(text_mask.sum()) < cfg["min_text_px"]:
        return None, None
    m = text_mask.astype(np.uint8)
    lim = cfg["max_angle"]
    coarse = np.arange(-lim, lim + 1e-6, cfg["coarse_step"])
    scores = [_profile_score(m, a) for a in coarse]
    best = float(coarse[int(np.argmax(scores))])
    fine = np.arange(best - cfg["coarse_step"], best + cfg["coarse_step"] + 1e-6, cfg["fine_step"])
    fscores = [_profile_score(m, a) for a in fine]
    best = float(fine[int(np.argmax(fscores))])
    base = _profile_score(m, 0.0)
    peak = max(fscores) / base if base > 0 else None
    return best, peak


def line_skew(ink: np.ndarray, work_dpi: float, cfg: dict) -> tuple:
    """Correction angle from long straight segments (form rules / dotted lines); (angle, n_segments, total_mm)."""
    px = lambda mm: mm_to_px(mm, work_dpi)
    segs = cv2.HoughLinesP(ink * 255, 1, np.pi / 1800, px(cfg["line_votes_mm"]),
                           minLineLength=px(cfg["line_min_mm"]), maxLineGap=px(cfg["line_gap_mm"]))
    if segs is None or len(segs) == 0:
        return None, 0, 0.0
    s = np.asarray(segs).reshape(-1, 4).astype(np.float64)     # OpenCV 4: Nx1x4, OpenCV 5: Nx4
    dx, dy = s[:, 2] - s[:, 0], s[:, 3] - s[:, 1]
    ang = np.degrees(np.arctan2(dy, dx))
    ang = (ang + 90) % 180 - 90                          # direction-free, in (-90, 90]
    keep = np.abs(ang) <= cfg["max_angle"]
    if keep.sum() < cfg["min_lines"]:
        return None, int(keep.sum()), 0.0
    ang, length = ang[keep], np.hypot(dx, dy)[keep]
    order = np.argsort(ang)
    cum = np.cumsum(length[order])
    median = float(ang[order][np.searchsorted(cum, cum[-1] / 2)])  # length-weighted median
    # image y points down: a segment sloping down-right (angle > 0) is straightened by rotating +angle in cv2
    return median, int(keep.sum()), float(length.sum() * 25.4 / work_dpi)


def warp_affine(img: np.ndarray, M: np.ndarray, size: tuple, border) -> np.ndarray:
    """warpAffine on the device's warp backend (OpenCL UMat / CUDA), CPU fallback."""
    backend = PROFILE["backends"]["warp"]
    try:
        if backend == "cuda":
            gm = cv2.cuda_GpuMat(); gm.upload(img)
            return cv2.cuda.warpAffine(gm, M, size, flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_CONSTANT,
                                       borderValue=border).download()
        if backend == "opencl":
            return cv2.warpAffine(cv2.UMat(img), M, size, flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_CONSTANT,
                                  borderValue=border).get()
    except cv2.error:
        pass
    return cv2.warpAffine(img, M, size, flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_CONSTANT, borderValue=border)


def rotate_expand(img: np.ndarray, angle: float, border) -> tuple:
    """Rotate about the centre by angle (cv2 convention), canvas grown so nothing is clipped. Returns (img, M)."""
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    nw, nh = int(np.ceil(w * cos + h * sin)), int(np.ceil(w * sin + h * cos))
    M[0, 2] += nw / 2 - w / 2
    M[1, 2] += nh / 2 - h / 2
    return warp_affine(img, M, (nw, nh), border), M


def transform_box(box: tuple, M: np.ndarray, size: tuple) -> tuple:
    x0, y0, x1, y1 = box
    pts = np.array([[x0, y0], [x1, y0], [x0, y1], [x1, y1]], np.float64)
    t = pts @ M[:, :2].T + M[:, 2]
    w, h = size
    return (int(max(0, np.floor(t[:, 0].min()))), int(max(0, np.floor(t[:, 1].min()))),
            int(min(w, np.ceil(t[:, 0].max()))), int(min(h, np.ceil(t[:, 1].max()))))


def estimate_skew(img: np.ndarray, box: tuple, dpi: float, cfg: dict) -> dict:
    """Combine the line and projection estimates by how trustworthy each one is (see CONFIG['deskew'])."""
    text, ink = deskew_masks(img, box, dpi, CONFIG["assess"], cfg["work_dpi"])
    p_angle, peak = projection_skew(text, cfg)
    l_angle, n_lines, line_mm = line_skew(ink, cfg["work_dpi"], cfg)
    lines_strong = l_angle is not None and n_lines >= cfg["lines_strong_min"] and line_mm >= cfg["lines_strong_mm"]
    proj_strong = p_angle is not None and peak is not None and peak >= cfg["projection_strong_peak"]
    close = (l_angle is not None and p_angle is not None and abs(l_angle - p_angle) <= cfg["agree_deg"])

    if lines_strong:
        angle, source = l_angle, "lines"
    elif proj_strong and (l_angle is None or abs(p_angle - l_angle) <= cfg["projection_conflict_deg"]):
        angle, source = p_angle, "projection"
    elif close:
        angle, source = l_angle, "both_weak_agree"
    else:
        angle, source = 0.0, "uncertain" if (l_angle is not None or p_angle is not None) else "none"
    disagree = lines_strong and proj_strong and not close
    suggested = max([abs(a) for a in (l_angle, p_angle) if a is not None], default=0.0)
    return {"angle": round(angle, 2), "source": source, "disagree": disagree,
            "uncertain": source in ("uncertain", "none") and suggested >= cfg["uncertain_flag_deg"],
            "projection_angle": None if p_angle is None else round(p_angle, 2),
            "projection_peak": None if peak is None else round(peak, 3),
            "line_angle": None if l_angle is None else round(l_angle, 2),
            "n_lines": n_lines, "line_length_mm": round(line_mm)}


def step5_deskew(rec: dict) -> None:
    if rec["metrics"].get(STEP5):                              # re-running the cell: start this step clean
        rec["flags"] = [f for f in rec["flags"] if f["step"] != STEP5]
        rec["actions"] = [a for a in rec["actions"] if a[0] != STEP5]
    cfg, review = CONFIG["deskew"], CONFIG["review"]["deskew"]
    img, box = rec["images"]["borders_clean"], rec["metrics"][STEP3]["content_box"]
    t0 = time.perf_counter()
    if rec["needs"]["skip_page"]:
        est = {"angle": 0.0, "source": "blank_not_checked", "disagree": False, "uncertain": False,
               "projection_angle": None, "projection_peak": None, "line_angle": None, "n_lines": 0, "line_length_mm": 0}
    else:
        est = estimate_skew(img, box, rec["dpi"], cfg)

    angle = est["angle"]
    if abs(angle) >= cfg["min_rotate_deg"]:
        border = tuple(float(v) for v in paper_bgr(rec["metrics"][STEP2]["paper_lab"]))
        out, M = rotate_expand(img, angle, border)
        new_box = transform_box(box, M, (out.shape[1], out.shape[0]))
        add_action(rec, STEP5, f"rotated {angle:+.2f}° ({est['source']}); canvas {img.shape[1]}x{img.shape[0]} -> "
                               f"{out.shape[1]}x{out.shape[0]}")
    else:
        out, new_box = img, box
        add_action(rec, STEP5, f"skew {angle:+.2f}° below {cfg['min_rotate_deg']}°: not rotated")
    rec["images"]["deskewed"] = out
    rec["metrics"][STEP5] = {**est, "rotated": out is not img, "content_box": new_box,
                             "orientation": "landscape" if out.shape[1] > out.shape[0] else "portrait",
                             "time_ms": round((time.perf_counter() - t0) * 1000)}

    if est["disagree"]:
        add_flag(rec, STEP5, "deskew_disagree", f"lines say {est['line_angle']:+.2f}°, text rows say "
                                                f"{est['projection_angle']:+.2f}° (both strong): used lines")
    if abs(angle) > review["large_skew_deg"]:
        add_flag(rec, STEP5, "large_skew", f"rotated {angle:+.2f}° ({est['source']}): slanted handwriting or real skew?")
    if est["uncertain"]:
        add_flag(rec, STEP5, "deskew_uncertain", f"no trustworthy estimate (lines {est['line_angle']}, text "
                                                 f"{est['projection_angle']} peak {est['projection_peak']}): not rotated")


run_parallel(step5_deskew, page_records, "Step 5 complete")

In [ ]:
# ---------- Step 5 review: angles, flags, before/after, optional self-test ----------
fmt = lambda v: "-" if v is None else f"{v:+.2f}"
print(f"{'Page':<16}{'Applied':>9}  {'Source':<11}{'Lines°':>8}{'#':>4}{'Text°':>8}{'Peak':>7}  {'Orient':<10}Flags")
for rec in page_records:
    m = rec["metrics"][STEP5]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP5) or "-"
    applied = f"{m['angle']:+.2f}°" if m["rotated"] else "-"
    print(f"{page_label(rec):<16}{applied:>9}  {m['source']:<11}{fmt(m['line_angle']):>8}{m['n_lines']:>4}"
          f"{fmt(m['projection_angle']):>8}{('-' if m['projection_peak'] is None else m['projection_peak']):>7}  "
          f"{m['orientation']:<10}{flags}")

rotated5 = [r for r in page_records if r["metrics"][STEP5]["rotated"]]
step5_flagged = [r for r in page_records if any(f["step"] == STEP5 for f in r["flags"])]
angles = [abs(r["metrics"][STEP5]["angle"]) for r in rotated5]
print(f"\nRotated {len(rotated5)} / {len(page_records)} pages"
      + (f" | |angle| median {np.median(angles):.2f}°, max {max(angles):.2f}°" if angles else ""))
print(f"Flagged for review in Step 5: {len(step5_flagged)}")
for rec in step5_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP5):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")

biggest5 = sorted(rotated5, key=lambda r: -abs(r["metrics"][STEP5]["angle"]))
show_before_after(step5_flagged + [r for r in biggest5 if r not in step5_flagged],
                  "borders_clean", "deskewed", "Step 5: deskew (flagged + largest angles)", STEP5)

# Self-test: tilt real pages by known angles and check the estimate (error = estimate + applied tilt)
RUN_DESKEW_SELF_TEST = False
if RUN_DESKEW_SELF_TEST:
    cfg = CONFIG["deskew"]
    errs = {"lines": [], "projection": []}
    for rec in [r for r in page_records if not r["needs"]["skip_page"]]:
        base = rec["images"]["deskewed"]
        for tilt in (-4.0, -1.5, -0.6, 0.6, 1.5, 4.0):
            tilted, M = rotate_expand(base, tilt, (255.0, 255.0, 255.0))
            box = transform_box(rec["metrics"][STEP5]["content_box"], M, (tilted.shape[1], tilted.shape[0]))
            est = estimate_skew(tilted, box, rec["dpi"], cfg)
            key = "lines" if est["source"] == "lines" else ("projection" if est["source"] == "projection" else "other")
            errs.setdefault(key, []).append(abs(est["angle"] + tilt))
    for src, e in errs.items():
        if e:
            e = np.array(e)
            print(f"Self-test [{src}]: {len(e)} cases | median error {np.median(e):.2f}° | 95th pct {np.percentile(e, 95):.2f}° "
                  f"| > 0.5°: {(e > 0.5).sum()}")

## Step 6: Illumination & paper-colour normalization

**Why:** shadows, yellowed paper and pink/lavender sheets lower contrast and make every page a different background. Later steps and any reader work better on white paper.

**Method (local white balance):**
1. **Paper colour everywhere:** on a 50 DPI copy, each colour channel is closed with a 12 mm window. This removes strokes, dense handwriting and black banner bars, which are all thinner than 12 mm. Median and Gaussian smoothing follow, then the estimate is resized back to full size.
2. **Divide it out per channel:** `out = image × 255 / paper`. Paper becomes white whatever its tint or shading. Ink colours are kept *relative to the paper*, so blue ink stays blue and purple stamps stay purple.
3. **Safety cap:** gain limited to 0.95–1.5×, so dark areas wider than the window (thumbprints, big stamps) are not bleached. If more than 1% of the page hits the cap, it is reported.
4. **Soft white point:** luminance 235–255 is eased to pure white (no hard threshold). Luminance, not the brightest channel: blue ink has a bright blue channel but is dark.

Applied to every non-blank page (`apply_to_all_pages`), so all pages share one white. Pages that were already white change by almost nothing.

**Verified per page (before → after):** paper lightness and tint, paper-level spread, stroke-pixel count (fading), and the blue/purple/red share of strokes. A share that *drops* is colour loss; a rise is expected, because ink on yellow or pink paper regains its true colour.

**Review flags:** `ink_loss`, `colour_loss`, `tint_remaining`, `illumination_remaining`, `gain_capped`.

**Safety (review):**
- Where the paper estimate is darker than 55% of the page's paper, it is an **object** (photo, thumbprint, seal, black banner), not paper. The estimate there is filled in (inpainted) from the surrounding paper, so the object keeps its tones; this is reported as `dark_object_kept`.
- The white point starts at luminance 245 (was 235), so pencil marks and very light stamps are not faded.

In [ ]:
STEP6 = "step6_illumination"


def estimate_paper(img: np.ndarray, dpi: float, cfg: dict) -> np.ndarray:
    """Per-channel paper colour at every pixel (float32, full size), and the fraction treated as dark objects."""
    h, w = img.shape[:2]
    f = cfg["bg_dpi"] / dpi
    small = cv2.resize(img, (max(1, int(w * f)), max(1, int(h * f))), interpolation=cv2.INTER_AREA)
    k = mm_to_px(cfg["bg_window_mm"], cfg["bg_dpi"], odd=True)
    s = mm_to_px(cfg["bg_smooth_mm"], cfg["bg_dpi"], odd=True, min_px=3)
    bg = cv2.morphologyEx(small, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (k, k)))
    bg = cv2.medianBlur(bg, s)
    # Dark OBJECTS wider than the window (photo, thumbprint, seal, black banner) survive the closing and would be
    # taken for dark paper and brightened. Where the estimate is far darker than the page's paper, fill it in from
    # the surrounding paper instead, so the object keeps its tones.
    bg_g = cv2.cvtColor(bg, cv2.COLOR_BGR2GRAY)
    paper_ref = float(np.percentile(bg_g, 90))
    obj = (bg_g < cfg["dark_object_rel"] * paper_ref).astype(np.uint8)
    obj_frac = float(obj.mean())
    if obj.any() and obj_frac < 0.9:
        obj = cv2.dilate(obj, np.ones((3, 3), np.uint8))
        bg = cv2.inpaint(bg, obj, max(3, k // 2), cv2.INPAINT_TELEA)
    bg = cv2.GaussianBlur(bg, (0, 0), s / 2)
    return cv2.resize(bg, (w, h), interpolation=cv2.INTER_LINEAR).astype(np.float32), obj_frac


def normalize_illumination(img: np.ndarray, dpi: float, cfg: dict) -> tuple:
    """Divide out the paper colour (per channel), soft white point. Returns (image, gain-capped frac, object frac)."""
    paper, obj_frac = estimate_paper(img, dpi, cfg)
    raw_gain = cfg["target"] / np.maximum(paper, 1.0)
    gain = np.clip(raw_gain, cfg["min_gain"], cfg["max_gain"])
    capped = float(np.mean(raw_gain.max(axis=2) > cfg["max_gain"]))
    out = img.astype(np.float32) * gain
    # Soft white point on luminance (not max channel: blue ink has a bright blue channel but is dark):
    # 0 at snap_start, 1 at 255, applied to all channels (keeps hue of ink)
    light = (0.114 * out[..., 0] + 0.587 * out[..., 1] + 0.299 * out[..., 2])[..., None]
    wgt = np.clip((light - cfg["snap_start"]) / max(1.0, 255.0 - cfg["snap_start"]), 0, 1)
    out = out * (1 - wgt) + 255.0 * wgt
    return np.clip(out + 0.5, 0, 255).astype(np.uint8), capped, obj_frac


def paper_ink_stats(img: np.ndarray, box: tuple, dpi: float, acfg: dict) -> dict:
    """Paper lightness/tint/spread and stroke count/colours inside the content box (for before/after checks)."""
    x0, y0, x1, y1 = box
    im = img[y0:y1, x0:x1]
    f = acfg["reference_dpi"] / dpi
    if abs(f - 1) > 0.05:
        im = cv2.resize(im, None, fx=f, fy=f, interpolation=cv2.INTER_AREA if f < 1 else cv2.INTER_CUBIC)
        dpi = acfg["reference_dpi"]
    g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    k = mm_to_px(acfg["stroke_width_max_mm"], dpi, odd=True)
    bh = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_RECT, (k, k)))
    ink = bh > acfg["stroke_ink"]
    pk = mm_to_px(acfg["paper_pad_mm"], dpi, odd=True)
    paper = ~cv2.dilate((bh > acfg["stroke_weak"]).astype(np.uint8), np.ones((pk, pk), np.uint8)).astype(bool)
    lab = cv2.cvtColor(im, cv2.COLOR_BGR2LAB)
    p_lab = np.median(lab[paper], axis=0) if paper.any() else np.array([255, 128, 128])
    q = 4
    sg = cv2.resize(g, (max(1, g.shape[1] // q), max(1, g.shape[0] // q)), interpolation=cv2.INTER_AREA)
    ck = mm_to_px(acfg["illum_close_mm"], dpi / q, odd=True)
    mk = mm_to_px(acfg["illum_median_mm"], dpi / q, odd=True, min_px=3)
    bg = cv2.medianBlur(cv2.morphologyEx(sg, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (ck, ck))), mk)
    lo, hi = acfg["illum_percentiles"]
    chroma = im.max(axis=2).astype(np.int16) - im.min(axis=2)
    hue = cv2.cvtColor(im, cv2.COLOR_BGR2HSV)[..., 0]
    colored = ink & (chroma >= acfg["ink_chroma_min"])
    n_ink = max(1, int(ink.sum()))

    def share(lo_hi):
        lo_h, hi_h = lo_hi
        band = (hue >= lo_h) & (hue < hi_h) if lo_h < hi_h else (hue >= lo_h) | (hue < hi_h)
        return round(float((colored & band).sum()) / n_ink * 100, 1)

    return {
        "paper_L": round(float(p_lab[0]) * 100 / 255, 1),
        "paper_tint": round(float(np.hypot(p_lab[1] - 128.0, p_lab[2] - 128.0)), 1),
        "illumination_spread": round(float(np.percentile(bg, hi) - np.percentile(bg, lo)), 1),
        "ink_px": int(ink.sum()),
        "ink_level": round(float(np.median(g[ink]))) if ink.any() else None,
        "ink_colors": {c: share(acfg["ink_hues"][c]) for c in ("blue", "purple", "red")},
    }


def step6_illumination(rec: dict) -> None:
    if rec["metrics"].get(STEP6):                              # re-running the cell: start this step clean
        rec["flags"] = [f for f in rec["flags"] if f["step"] != STEP6]
        rec["actions"] = [a for a in rec["actions"] if a[0] != STEP6]
    cfg, review = CONFIG["illumination"], CONFIG["review"]["illumination"]
    img, box = rec["images"]["deskewed"], rec["metrics"][STEP5]["content_box"]
    needed = bool(rec["needs"]["illumination"] or rec["needs"]["tint_removal"])
    if rec["needs"]["skip_page"] or not (cfg["apply_to_all_pages"] or needed):
        rec["images"]["illum_norm"] = img
        rec["metrics"][STEP6] = {"applied": False, "needed": needed}
        add_action(rec, STEP6, "not applied")
        return

    t0 = time.perf_counter()
    out, capped, obj_frac = normalize_illumination(img, rec["dpi"], cfg)
    before = paper_ink_stats(img, box, rec["dpi"], CONFIG["assess"])
    after = paper_ink_stats(out, box, rec["dpi"], CONFIG["assess"])
    rec["images"]["illum_norm"] = out
    ink_change = (after["ink_px"] - before["ink_px"]) / max(1, before["ink_px"]) * 100
    colour_loss = max(before["ink_colors"][c] - after["ink_colors"][c] for c in ("blue", "purple", "red"))
    rec["metrics"][STEP6] = {"applied": True, "needed": needed, "before": before, "after": after,
                             "ink_change_pct": round(ink_change, 1), "colour_loss_pts": round(colour_loss, 1),
                             "gain_capped_pct": round(capped * 100, 2), "dark_object_pct": round(obj_frac * 100, 2),
                             "time_ms": round((time.perf_counter() - t0) * 1000)}
    add_action(rec, STEP6, f"paper L {before['paper_L']} -> {after['paper_L']}, tint {before['paper_tint']} -> "
                           f"{after['paper_tint']}, spread {before['illumination_spread']} -> "
                           f"{after['illumination_spread']}" + ("" if needed else " (not needed; consistency)"))

    if -ink_change > review["ink_loss_pct"]:
        add_flag(rec, STEP6, "ink_loss", f"stroke pixels {ink_change:+.1f}% after normalization")
    if colour_loss > review["colour_loss_pts"]:
        add_flag(rec, STEP6, "colour_loss", f"ink colour share dropped by {colour_loss:.1f} points "
                                            f"({before['ink_colors']} -> {after['ink_colors']})")
    if after["paper_tint"] >= review["tint_remaining"]:
        add_flag(rec, STEP6, "tint_remaining", f"paper tint still {after['paper_tint']}")
    if after["illumination_spread"] >= review["illumination_remaining"]:
        add_flag(rec, STEP6, "illumination_remaining", f"paper-level spread still {after['illumination_spread']}")
    if obj_frac * 100 > review["dark_object_pct"]:
        add_flag(rec, STEP6, "dark_object_kept", f"{obj_frac * 100:.1f}% of the page is a dark object (photo, thumbprint, "
                                                 "seal, banner): its tones were kept, not brightened")
    if capped * 100 > review["gain_capped_pct"]:
        add_flag(rec, STEP6, "gain_capped", f"{capped * 100:.1f}% of the page needed more than "
                                            f"{cfg['max_gain']}x gain (large dark area or very dark paper): left darker")


run_parallel(step6_illumination, page_records, "Step 6 complete")

In [ ]:
# ---------- Step 6 review: before -> after per page, flags, before/after images ----------
print(f"{'Page':<16}{'Need':>5}{'Paper L':>15}{'Tint':>13}{'Spread':>13}{'Ink Δ%':>8}{'Col -':>7}{'Cap %':>7}  Flags")
for rec in page_records:
    m = rec["metrics"][STEP6]
    if not m["applied"]:
        print(f"{page_label(rec):<16}{'-':>5}  not applied")
        continue
    b, a = m["before"], m["after"]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP6) or "-"
    print(f"{page_label(rec):<16}{('yes' if m['needed'] else '-'):>5}{b['paper_L']:>7}->{a['paper_L']:<6}"
          f"{b['paper_tint']:>6}->{a['paper_tint']:<5}{b['illumination_spread']:>6}->{a['illumination_spread']:<5}"
          f"{m['ink_change_pct']:>8}{m['colour_loss_pts']:>7}{m['gain_capped_pct']:>7}  {flags}")

applied6 = [r for r in page_records if r["metrics"][STEP6]["applied"]]
if applied6:
    for key in ("paper_L", "paper_tint", "illumination_spread"):
        b = [r["metrics"][STEP6]["before"][key] for r in applied6]
        a = [r["metrics"][STEP6]["after"][key] for r in applied6]
        print(f"{key:<20} median {np.median(b):.1f} -> {np.median(a):.1f} | worst {max(b):.1f} -> {max(a):.1f}")
step6_flagged = [r for r in page_records if any(f["step"] == STEP6 for f in r["flags"])]
print(f"Flagged for review in Step 6: {len(step6_flagged)}")
for rec in step6_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP6):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")

# Flagged pages first, then the pages that changed most (tint / lighting)
changed6 = sorted(applied6, key=lambda r: -(r["metrics"][STEP6]["before"]["paper_tint"]
                                           + r["metrics"][STEP6]["before"]["illumination_spread"]))
show_before_after(step6_flagged + [r for r in changed6 if r not in step6_flagged], "deskewed", "illum_norm",
                  "Step 6: illumination & paper colour (flagged + biggest changes)", STEP6)

## Step 7: Noise removal (adaptive)

**Why:** photocopy grain, grey toner texture, grain amplified when Step 4 flattened spine shadows, colour speckle from JPEG, and isolated specks. Thin handwriting must survive.

**Detect on the current image:** Step 2's `assess_page` runs again on `images["illum_norm"]`, because the Step 2 numbers predate the Step 4 flattening. Each fix runs only where its measurement says so:

| Found | Fix |
|---|---|
| `noise_sigma` ≥ 1.0 (mild) / ≥ 2.0 (strong), or `texture_pct` ≥ 2.0 | Non-local means on **lightness only** (Lab L), h = 6 / 12, patch 0.9 mm, search 2.7 mm. Device backend: OpenCL / CUDA / CPU. |
| any denoise on a colour page | 0.6 mm median on Lab a/b (colour speckle; ink colour spans more pixels) |
| `texture_pct` ≥ 0.5 | Faint **dot-like** marks away from real strokes (the blobs Step 2 counts as texture) → paper. Ruled/dotted lines and faint **stroke-shaped** marks are kept (Step 9 handles bleed-through). |
| `isolated_specks_per_mp` ≥ 100 | Blobs ≤ 0.1 mm² with no other ink within ~2 mm → paper. Dotted lines, full stops and matras all have neighbours, so they stay. |

**Verify:** `assess_page` runs again after the fix: noise, texture, specks and ink before → after.

**Review flags:** `ink_loss` (pixels of real strokes, connected ink ≥ 0.5 mm², dropped > 8%; grain is excluded because removing it is the point), `noise_remaining`, `specks_remaining`.

**Known limit:** coarse grain left in spine shadows that Step 4 flattened (FIR-22-2010 p1/p3) is not removed. Non-local means leaves it, and removing small clusters inside the flattened zone was tried and reverted: the grain joins into larger clusters and survives, while the dots of dotted fill-in lines next to the spine were erased. Those pages stay flagged `noise_remaining`. The zoom view shows a full-resolution crop of the area that changed most on each page.

**Safety (review):**
- Texture and speck clearing only ever touches **neutral grey** marks; texture also only **faint** ones (black-hat ≤ 30). Stamps, seals and coloured ink dots are never erased.
- The colour median is applied to **paper pixels only**, so thin coloured strokes keep their colour.

In [ ]:
STEP7 = "step7_denoise"


def stroke_and_faint_masks(g: np.ndarray, dpi: float, acfg: dict) -> tuple:
    """Same definitions as Step 2: (ink, texture dots, isolated specks, faint stroke-like marks) on a grey image."""
    px = lambda mm, odd=False: mm_to_px(mm, dpi, odd)
    sk = px(acfg["stroke_width_max_mm"], odd=True)
    bh = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_RECT, (sk, sk)))
    weak, ink, strong = bh > acfg["stroke_weak"], bh > acfg["stroke_ink"], bh > acfg["stroke_strong"]
    wk = weak.astype(np.uint8)
    gap, run_len, pad = px(acfg["rule_line_gap_mm"], odd=True), px(acfg["rule_line_min_mm"], odd=True), px(acfg["rule_line_pad_mm"], odd=True)
    h_lines = cv2.morphologyEx(cv2.dilate(wk, np.ones((1, gap), np.uint8)), cv2.MORPH_OPEN, np.ones((1, run_len), np.uint8))
    v_lines = cv2.morphologyEx(cv2.dilate(wk, np.ones((gap, 1), np.uint8)), cv2.MORPH_OPEN, np.ones((run_len, 1), np.uint8))
    lines = cv2.dilate(h_lines | v_lines, np.ones((pad, pad), np.uint8)).astype(bool)
    nk = px(acfg["near_ink_mm"], odd=True)
    near_strong = cv2.dilate(strong.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (nk, nk))).astype(bool)
    faint = (weak & ~near_strong & ~lines).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(faint, connectivity=8)
    fa, fw, fh = st[:, cv2.CC_STAT_AREA], st[:, cv2.CC_STAT_WIDTH], st[:, cv2.CC_STAT_HEIGHT]
    mm2_px = (dpi / 25.4) ** 2
    stroke_like = ((fa >= acfg["faint_stroke_min_area_mm2"] * mm2_px) & (np.minimum(fw, fh) >= px(acfg["faint_stroke_min_size_mm"]))
                   & (fa / np.maximum(1, fw * fh) < acfg["faint_stroke_max_fill"]))
    stroke_like[0] = True                                      # background label
    texture = ~stroke_like[lab]
    faint_strokes = stroke_like[lab] & (lab > 0)                # bleed-through candidates (Step 9)
    # isolated specks among ink
    _, lab2, st2, cen = cv2.connectedComponentsWithStats(ink.astype(np.uint8), connectivity=8)
    win = px(acfg["speck_isolation_mm"], odd=True)
    box = cv2.boxFilter(ink.astype(np.float32), -1, (win, win), normalize=False)
    cx = np.clip(np.rint(cen[:, 0]).astype(int), 0, g.shape[1] - 1)
    cy = np.clip(np.rint(cen[:, 1]).astype(int), 0, g.shape[0] - 1)
    a2 = st2[:, cv2.CC_STAT_AREA]
    is_speck = (a2 <= acfg["speck_max_area_mm2"] * mm2_px) & (box[cy, cx] <= a2 + 0.5)
    is_speck[0] = False
    specks = is_speck[lab2]
    return ink, texture, specks, faint_strokes


def stroke_pixels(img: np.ndarray, dpi: float, min_mm2: float, acfg: dict) -> int:
    """Ink pixels in connected strokes of at least min_mm2 (grain and specks excluded)."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sk = mm_to_px(acfg["stroke_width_max_mm"], dpi, odd=True)
    bh = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_RECT, (sk, sk)))
    ink = (bh > acfg["stroke_ink"]).astype(np.uint8)
    _, _, st, _ = cv2.connectedComponentsWithStats(ink, connectivity=8)
    a = st[1:, cv2.CC_STAT_AREA]
    return int(a[a >= min_mm2 * (dpi / 25.4) ** 2].sum())


def nlm_luminance(L: np.ndarray, h: float, tw: int, sw: int) -> np.ndarray:
    backend = PROFILE["backends"]["denoise"]
    try:
        if backend == "cuda":
            gm = cv2.cuda_GpuMat(); gm.upload(L)
            return cv2.cuda.fastNlMeansDenoising(gm, h, search_window=sw, block_size=tw).download()
        if backend == "opencl":
            return cv2.fastNlMeansDenoising(cv2.UMat(L), None, h, tw, sw).get()
    except cv2.error:
        pass
    return cv2.fastNlMeansDenoising(L, None, h, tw, sw)


def denoise_page(img: np.ndarray, dpi: float, plan: dict, cfg: dict, acfg: dict) -> tuple:
    """Apply the planned fixes. Returns (image, pixels changed by texture cleanup, by speck removal)."""
    px = lambda mm, odd=False: mm_to_px(mm, dpi, odd)
    out = img
    if plan["nlm"] != "none":
        lab = cv2.cvtColor(out, cv2.COLOR_BGR2LAB)
        L, a, b = cv2.split(lab)
        tw, sw = px(cfg["nlm_template_mm"], odd=True), px(cfg["nlm_search_mm"], odd=True)
        L = nlm_luminance(L, cfg["nlm_h"][plan["nlm"]], tw, sw)
        if plan["chroma"]:                                       # colour speckle on PAPER only: ink keeps its colour
            ck = px(cfg["chroma_median_mm"], odd=True)
            sk = px(acfg["stroke_width_max_mm"], odd=True)
            bh_l = cv2.morphologyEx(L, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_RECT, (sk, sk)))
            ink_px = cv2.dilate((bh_l > acfg["stroke_weak"]).astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)
            a = np.where(ink_px, a, cv2.medianBlur(a, ck))
            b = np.where(ink_px, b, cv2.medianBlur(b, ck))
        out = cv2.cvtColor(cv2.merge([L, a, b]), cv2.COLOR_LAB2BGR)
    n_tex = n_spk = 0
    if plan["texture"] or plan["specks"]:
        g = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)
        _, texture, specks, _ = stroke_and_faint_masks(g, dpi, acfg)
        # Only NEUTRAL (grey) marks are ever cleared: toner grain and scanner specks are grey; stamps, seals and
        # coloured ink dots are not. Texture clearing is further limited to faint marks (black-hat <= stroke_ink).
        neutral = (out.max(axis=2).astype(np.int16) - out.min(axis=2)) < acfg["ink_chroma_min"]
        sk = px(acfg["stroke_width_max_mm"], odd=True)
        faint = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_RECT, (sk, sk))) <= acfg["stroke_ink"]
        texture &= neutral & faint
        specks &= neutral
        clear = np.zeros(g.shape, bool)
        if plan["texture"]:
            clear |= texture; n_tex = int(texture.sum())
        if plan["specks"]:
            clear |= specks; n_spk = int(specks.sum())
        if clear.any():
            out = out.copy()
            out[clear] = cfg["texture_target"]
    return out, n_tex, n_spk


def step7_denoise(rec: dict) -> None:
    if rec["metrics"].get(STEP7):                              # re-running the cell: start this step clean
        rec["flags"] = [f for f in rec["flags"] if f["step"] != STEP7]
        rec["actions"] = [a for a in rec["actions"] if a[0] != STEP7]
    cfg, review, need = CONFIG["denoise"], CONFIG["review"]["denoise"], CONFIG["assess"]["needs"]
    img = rec["images"]["illum_norm"]
    if rec["needs"]["skip_page"]:
        rec["images"]["denoised"] = img
        rec["metrics"][STEP7] = {"applied": False}
        return
    t0 = time.perf_counter()
    before = assess_page(img, CONFIG["assess"], rec["dpi"])
    if before["noise_sigma"] >= need["denoise_strong_sigma"]:
        nlm = "strong"
    elif before["noise_sigma"] >= need["denoise_mild_sigma"] or before["texture_pct"] >= need["denoise_texture_pct"]:
        nlm = "mild"
    else:
        nlm = "none"
    is_colour = not rec["metrics"][STEP1]["greyscale_scan"]
    plan = {"nlm": nlm, "chroma": nlm != "none" and is_colour,
            "texture": before["texture_pct"] >= cfg["texture_cleanup_pct"],
            "specks": before["isolated_specks_per_mp"] >= need["speck_removal_per_mp"]}
    if not any([plan["nlm"] != "none", plan["texture"], plan["specks"]]):
        rec["images"]["denoised"] = img
        rec["metrics"][STEP7] = {"applied": False, "plan": plan, "before": {k: before[k] for k in
                                 ("noise_sigma", "texture_pct", "isolated_specks_per_mp", "ink_pct")}}
        add_action(rec, STEP7, "no noise to remove")
        return

    out, n_tex, n_spk = denoise_page(img, rec["dpi"], plan, cfg, CONFIG["assess"])
    after = assess_page(out, CONFIG["assess"], rec["dpi"])
    keys = ("noise_sigma", "texture_pct", "isolated_specks_per_mp", "ink_pct")
    s_before = stroke_pixels(img, rec["dpi"], cfg["stroke_min_mm2"], CONFIG["assess"])
    s_after = stroke_pixels(out, rec["dpi"], cfg["stroke_min_mm2"], CONFIG["assess"])
    ink_change = (s_after - s_before) / max(1, s_before) * 100                 # real strokes only
    rec["images"]["denoised"] = out
    rec["metrics"][STEP7] = {"applied": True, "plan": plan, "before": {k: before[k] for k in keys},
                             "after": {k: after[k] for k in keys}, "ink_change_pct": round(ink_change, 1),
                             "texture_px_cleared": n_tex, "speck_px_cleared": n_spk,
                             "time_ms": round((time.perf_counter() - t0) * 1000)}
    done = [f"NLM {nlm}"] if nlm != "none" else []
    done += ["chroma median"] if plan["chroma"] else []
    done += [f"texture dots cleared ({n_tex} px)"] if plan["texture"] else []
    done += [f"isolated specks cleared ({n_spk} px)"] if plan["specks"] else []
    add_action(rec, STEP7, "; ".join(done) + f" | noise {before['noise_sigma']} -> {after['noise_sigma']}, "
                           f"texture {before['texture_pct']} -> {after['texture_pct']}%, "
                           f"specks {before['isolated_specks_per_mp']} -> {after['isolated_specks_per_mp']}/MP")
    if -ink_change > review["ink_loss_pct"]:
        add_flag(rec, STEP7, "ink_loss", f"real-stroke pixels {ink_change:+.1f}% (all ink {before['ink_pct']}% -> {after['ink_pct']}%)")
    if after["noise_sigma"] >= review["noise_remaining_sigma"]:
        add_flag(rec, STEP7, "noise_remaining", f"noise sigma still {after['noise_sigma']}")
    if after["isolated_specks_per_mp"] >= review["specks_remaining_per_mp"]:
        add_flag(rec, STEP7, "specks_remaining", f"{after['isolated_specks_per_mp']} isolated specks/MP remain")


run_parallel(step7_denoise, page_records, "Step 7 complete")

In [ ]:
# ---------- Step 7 review: plan + before -> after, flags, zoomed crops where the change was largest ----------
print(f"{'Page':<16}{'NLM':<8}{'Chr':>4}{'Tex':>4}{'Spk':>4}{'Noise':>14}{'Texture %':>14}{'Specks/MP':>13}{'All ink %':>14}{'Strokes Δ%':>11}  Flags")
for rec in page_records:
    m = rec["metrics"][STEP7]
    if not m.get("applied"):
        continue
    p, b, a = m["plan"], m["before"], m["after"]
    yn = lambda v: "y" if v else "-"
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP7) or "-"
    print(f"{page_label(rec):<16}{p['nlm']:<8}{yn(p['chroma']):>4}{yn(p['texture']):>4}{yn(p['specks']):>4}"
          f"{b['noise_sigma']:>7}->{a['noise_sigma']:<6}{b['texture_pct']:>7}->{a['texture_pct']:<6}"
          f"{b['isolated_specks_per_mp']:>6}->{a['isolated_specks_per_mp']:<6}{b['ink_pct']:>7}->{a['ink_pct']:<6}"
          f"{m['ink_change_pct']:>11}  {flags}")
applied7 = [r for r in page_records if r["metrics"][STEP7].get("applied")]
print(f"\nDenoised {len(applied7)} / {len(page_records)} pages "
      f"(strong {sum(r['metrics'][STEP7]['plan']['nlm'] == 'strong' for r in applied7)}, "
      f"mild {sum(r['metrics'][STEP7]['plan']['nlm'] == 'mild' for r in applied7)}, "
      f"texture {sum(r['metrics'][STEP7]['plan']['texture'] for r in applied7)}, "
      f"specks {sum(r['metrics'][STEP7]['plan']['specks'] for r in applied7)})")
step7_flagged = [r for r in page_records if any(f["step"] == STEP7 for f in r["flags"])]
print(f"Flagged for review in Step 7: {len(step7_flagged)}")
for rec in step7_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP7):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")


def show_zoom_pairs(records: list[dict], before: str, after: str, title: str, step: str,
                    crop_mm: float = 45, max_pages: int = 8) -> None:
    """Full-resolution before|after crops centred where the two images differ most (noise is invisible in thumbnails)."""
    records = records[:max_pages]
    if not records:
        return
    fig, axes = plt.subplots(len(records), 1, figsize=(14, 4.2 * len(records)), squeeze=False)
    for ax, rec in zip(axes[:, 0], records):
        a, b = rec["images"][before], rec["images"][after]
        if a.shape != b.shape:
            ax.axis("off"); continue
        diff = cv2.absdiff(cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)).astype(np.float32)
        c = mm_to_px(crop_mm, rec["dpi"])
        score = cv2.boxFilter(diff, -1, (c, c))
        y, x = np.unravel_index(int(np.argmax(score)), score.shape)
        y0 = int(np.clip(y - c // 2, 0, max(0, a.shape[0] - c))); x0 = int(np.clip(x - c, 0, max(0, a.shape[1] - 2 * c)))
        crop = lambda im: im[y0:y0 + c, x0:x0 + 2 * c]
        ax.imshow(cv2.cvtColor(np.hstack([crop(a), np.full((crop(a).shape[0], 8, 3), 0, np.uint8), crop(b)]), cv2.COLOR_BGR2RGB))
        codes = ", ".join(f["code"] for f in rec["flags"] if f["step"] == step)
        ax.set_title(f"{page_label(rec)}  (before | after, 1:1 crop)" + (f"  ⚑ {codes}" if codes else ""), fontsize=10,
                     color="red" if codes else "black")
        ax.axis("off")
    fig.suptitle(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


most7 = sorted(applied7, key=lambda r: -(r["metrics"][STEP7]["before"]["noise_sigma"] + r["metrics"][STEP7]["before"]["texture_pct"]))
show_zoom_pairs(step7_flagged + [r for r in most7 if r not in step7_flagged], "illum_norm", "denoised",
                "Step 7: noise removal, zoomed where the change is largest", STEP7)

## Step 8: Deblur / sharpen (adaptive)

**Why:** some pages are soft: out-of-focus scans, or ink that has spread into the paper. Sharpening helps reading, but it must not sharpen noise or draw halos around strokes, and it must not touch pages that are already sharp.

**Detect on the current image:** `assess_page` again on `images["denoised"]` (sharpness = high-frequency energy on stroke edges ÷ stroke contrast, measured at 200 DPI). Sharpen only if sharpness < `sharpen_below` (7.0).

**Method (unsharp mask on lightness only):**
- `detail = L − Gaussian(L, 0.2 mm)`.
- Detail smaller than 4 grey levels is not boosted (paper texture, residual noise).
- **Strength from the deficit:** `amount = 3 × (9 − sharpness) / 9`, clipped to 0.3–1.5, so a slightly soft page gets a light touch.
- **Halo clamp:** each pixel stays within its 3×3 neighbourhood's min/max ± 8 grey levels.
- Lab a/b (ink colour) untouched.

**Verify:** sharpness, noise and real-stroke pixels before → after.

**Review flags:** `still_blurry` (< 5.0 after), `noise_amplified` (σ +0.5), `ink_change` (real strokes ±10%).

**Self-test (optional):** blur sharp pages by a known amount, sharpen, and compare.

In [ ]:
STEP8 = "step8_sharpen"


def sharpen_luminance(img: np.ndarray, dpi: float, amount: float, cfg: dict) -> np.ndarray:
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    L = lab[..., 0].astype(np.float32)
    detail = L - cv2.GaussianBlur(L, (0, 0), max(0.5, cfg["sigma_mm"] * dpi / 25.4))
    t = cfg["detail_threshold"]
    weight = np.clip((np.abs(detail) - t) / max(1.0, t), 0, 1)           # soft threshold: no boost below t
    sharp = L + amount * detail * weight
    k = np.ones((3, 3), np.uint8)
    lo = cv2.erode(lab[..., 0], k).astype(np.float32) - cfg["halo_overshoot"]
    hi = cv2.dilate(lab[..., 0], k).astype(np.float32) + cfg["halo_overshoot"]
    lab[..., 0] = np.clip(np.clip(sharp, lo, hi) + 0.5, 0, 255).astype(np.uint8)
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def sharpen_amount(sharpness: float, cfg: dict) -> float:
    return float(np.clip(cfg["gain"] * (cfg["target_sharpness"] - sharpness) / cfg["target_sharpness"],
                         cfg["min_amount"], cfg["max_amount"]))


def step8_sharpen(rec: dict) -> None:
    if rec["metrics"].get(STEP8):                              # re-running the cell: start this step clean
        rec["flags"] = [f for f in rec["flags"] if f["step"] != STEP8]
        rec["actions"] = [a for a in rec["actions"] if a[0] != STEP8]
    cfg, review = CONFIG["sharpen"], CONFIG["review"]["sharpen"]
    img = rec["images"]["denoised"]
    if rec["needs"]["skip_page"]:
        rec["images"]["sharpened"] = img
        rec["metrics"][STEP8] = {"applied": False}
        return
    t0 = time.perf_counter()
    before = assess_page(img, CONFIG["assess"], rec["dpi"])
    if before["sharpness"] >= CONFIG["assess"]["needs"]["sharpen_below"]:
        rec["images"]["sharpened"] = img
        rec["metrics"][STEP8] = {"applied": False, "sharpness": before["sharpness"]}
        add_action(rec, STEP8, f"sharp enough ({before['sharpness']})")
        return

    amount = sharpen_amount(before["sharpness"], cfg)
    out = sharpen_luminance(img, rec["dpi"], amount, cfg)
    after = assess_page(out, CONFIG["assess"], rec["dpi"])
    s_b = stroke_pixels(img, rec["dpi"], CONFIG["denoise"]["stroke_min_mm2"], CONFIG["assess"])
    s_a = stroke_pixels(out, rec["dpi"], CONFIG["denoise"]["stroke_min_mm2"], CONFIG["assess"])
    stroke_change = (s_a - s_b) / max(1, s_b) * 100
    rec["images"]["sharpened"] = out
    rec["metrics"][STEP8] = {"applied": True, "amount": round(amount, 2),
                             "sharpness": [before["sharpness"], after["sharpness"]],
                             "noise_sigma": [before["noise_sigma"], after["noise_sigma"]],
                             "stroke_change_pct": round(stroke_change, 1),
                             "time_ms": round((time.perf_counter() - t0) * 1000)}
    add_action(rec, STEP8, f"unsharp amount {amount:.2f}: sharpness {before['sharpness']} -> {after['sharpness']}, "
                           f"noise {before['noise_sigma']} -> {after['noise_sigma']}")
    if after["sharpness"] < review["still_blurry_below"]:
        add_flag(rec, STEP8, "still_blurry", f"sharpness {after['sharpness']} after sharpening")
    if after["noise_sigma"] - before["noise_sigma"] > review["noise_rise_sigma"]:
        add_flag(rec, STEP8, "noise_amplified", f"noise sigma {before['noise_sigma']} -> {after['noise_sigma']}")
    if abs(stroke_change) > review["stroke_change_pct"]:
        add_flag(rec, STEP8, "ink_change", f"real-stroke pixels {stroke_change:+.1f}%")


run_parallel(step8_sharpen, page_records, "Step 8 complete")

In [ ]:
# ---------- Step 8 review: sharpness of every page, what was sharpened, zoom, optional self-test ----------
sharp_now = sorted(((r["metrics"][STEP8]["sharpness"] if not r["metrics"][STEP8].get("applied")
                     else r["metrics"][STEP8]["sharpness"][0]), page_label(r))
                   for r in page_records if "sharpness" in r["metrics"][STEP8])
print("Sharpness before Step 8 (lowest first):", ", ".join(f"{lbl} {s}" for s, lbl in sharp_now[:10]), "...")
applied8 = [r for r in page_records if r["metrics"][STEP8].get("applied")]
print(f"\n{'Page':<16}{'Amount':>7}{'Sharpness':>16}{'Noise':>14}{'Strokes Δ%':>11}  Flags")
for rec in applied8:
    m = rec["metrics"][STEP8]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP8) or "-"
    print(f"{page_label(rec):<16}{m['amount']:>7}{m['sharpness'][0]:>8}->{m['sharpness'][1]:<6}"
          f"{m['noise_sigma'][0]:>7}->{m['noise_sigma'][1]:<6}{m['stroke_change_pct']:>11}  {flags}")
print(f"\nSharpened {len(applied8)} / {len(page_records)} pages")
step8_flagged = [r for r in page_records if any(f["step"] == STEP8 for f in r["flags"])]
print(f"Flagged for review in Step 8: {len(step8_flagged)}")
for rec in step8_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP8):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")
show_zoom_pairs(step8_flagged + [r for r in applied8 if r not in step8_flagged], "denoised", "sharpened",
                "Step 8: sharpening, zoomed where the change is largest", STEP8, crop_mm=35)

# Self-test: blur sharp pages by a known Gaussian, sharpen, compare sharpness (original -> blurred -> restored)
RUN_SHARPEN_SELF_TEST = False
if RUN_SHARPEN_SELF_TEST:
    cfg = CONFIG["sharpen"]
    rows = []
    for rec in [r for r in page_records if not r["needs"]["skip_page"]][::4]:
        base = rec["images"]["sharpened"]
        s0 = assess_page(base, CONFIG["assess"], rec["dpi"])["sharpness"]
        for sigma_mm in (0.1, 0.2):
            blurred = cv2.GaussianBlur(base, (0, 0), sigma_mm * rec["dpi"] / 25.4)
            sb = assess_page(blurred, CONFIG["assess"], rec["dpi"])["sharpness"]
            restored = sharpen_luminance(blurred, rec["dpi"], sharpen_amount(sb, cfg), cfg) if sb < CONFIG["assess"]["needs"]["sharpen_below"] else blurred
            sr = assess_page(restored, CONFIG["assess"], rec["dpi"])["sharpness"]
            rows.append((page_label(rec), sigma_mm, s0, sb, sr))
    for lbl, sg, s0, sb, sr in rows:
        print(f"  {lbl:<16} blur {sg} mm: {s0} -> blurred {sb} -> restored {sr}")
    rec_frac = [(sr - sb) / max(0.01, s0 - sb) for _, _, s0, sb, sr in rows if s0 - sb > 0.5]
    if rec_frac:
        print(f"Self-test: median recovery of lost sharpness {np.median(rec_frac) * 100:.0f}%")

## Step 9: Bleed-through suppression

**Why:** writing from the back of the sheet shows through (FIR-2-2009 p8, p9, p5) and competes with the real text.

**What bleed-through looks like here:** the mirrored back-side writing is not "faint" by the stroke measure. It has clear edges, and it is the **same ink colour as the front writing, only lighter**. Front blue ink on p8 has a mean lightness of about 174; the show-through is about 214.

**Detect (every page):**
1. Ink components (black-hat strokes ≥ 0.3 mm²) are grouped by colour: blue / purple / red (hue bands from `CONFIG["assess"]["ink_hues"]`).
2. In each group, component lightness is split with Otsu. It counts as bleed-through only if the split is clean: separability η ≥ 0.8, gap ≥ 30 L, light share 25–85%, ≥ 100 components.
3. Black/grey ink is **not** tested: printed text and grey dotted lines always split, even on clean forms.

On the samples: bleed pages η 0.88–0.92 with a gap of 39–58. Normal blue pages η 0.47–0.69, with a gap of 12–16 or a light share ≤ 14%.

**Fix (per pixel, not per component):**
- Pixels of that colour that are **lighter than the split** are blended **85% toward paper**: lightened, not erased.
- Pixels within 0.25 mm of **dark ink of that colour** (front strokes and their soft edges) or of **black/grey ink** (print) are never touched.

*Rejected:* classifying whole connected components. Bleed touches front strokes and print, so they merged into one "light blue" component. That lightened a dark front stroke on FIR-2-2009 p8 and erased printed text on p9.

**Review flags:** **every page this step changes is flagged `bleed_suppressed`**. Also `ink_change` if the dark (front) strokes changed by more than 5%.

**Limits:**
- Show-through that overlaps the front writing is part of the same components and stays.
- Black-ink bleed-through is not handled.
- Step 2's faint-stroke trigger is reported for comparison only.

**Safety (review):** lightening is 60% (was 85%). Show-through fades, but if real lighter writing were ever misdetected it stays readable.

In [ ]:
STEP9 = "step9_bleed"


def ink_components(img: np.ndarray, dpi: float, cfg: dict, acfg: dict) -> tuple:
    """Ink components with their area, mean lightness, mean chroma and mean hue."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sk = mm_to_px(acfg["stroke_width_max_mm"], dpi, odd=True)
    bh = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_RECT, (sk, sk)))
    ink = (bh > acfg["stroke_ink"]).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink, connectivity=8)
    idx = lab.ravel()
    m = idx > 0
    L = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)[..., 0].ravel().astype(np.float64)
    chroma = (img.max(axis=2).astype(np.int16) - img.min(axis=2)).ravel().astype(np.float64)
    hue = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[..., 0].ravel().astype(np.float64)
    area = np.bincount(idx[m], minlength=n).astype(np.float64)
    mean = lambda v: np.bincount(idx[m], weights=v[m], minlength=n) / np.maximum(area, 1)
    comps = {"area": area, "L": mean(L), "chroma": mean(chroma), "hue": mean(hue)}
    big = area >= cfg["min_component_mm2"] * (dpi / 25.4) ** 2
    big[0] = False
    return lab, comps, big


def otsu_split(values: np.ndarray, weights: np.ndarray) -> dict:
    """Weighted Otsu split of 1-D values: threshold, separability eta, class means, light share."""
    order = np.argsort(values)
    v, w = values[order], weights[order]
    cw, cwv = np.cumsum(w), np.cumsum(w * v)
    tot, totv = cw[-1], cwv[-1]
    w0, w1 = cw[:-1], tot - cw[:-1]
    ok = (w0 > 0) & (w1 > 0)
    m0 = np.where(ok, cwv[:-1] / np.maximum(w0, 1e-9), 0)
    m1 = np.where(ok, (totv - cwv[:-1]) / np.maximum(w1, 1e-9), 0)
    between = np.where(ok, w0 * w1 * (m0 - m1) ** 2, -1)
    i = int(np.argmax(between))
    mean = totv / tot
    var = float((w * (v - mean) ** 2).sum() / tot)
    eta = float(between[i] / tot ** 2 / max(var, 1e-9))
    return {"threshold": float((v[i] + v[i + 1]) / 2), "eta": eta, "dark_mean": float(m0[i]),
            "light_mean": float(m1[i]), "light_share": float(w1[i] / tot)}


def detect_bleed(img: np.ndarray, dpi: float, cfg: dict, acfg: dict) -> tuple:
    """Per colour group: is component lightness split into a clean dark (front) and light (show-through) population?
    Returns (labels, light-population component ids, per-group stats)."""
    lab, comps, big = ink_components(img, dpi, cfg, acfg)
    hues = acfg["ink_hues"]
    light_ids, stats = [], {}
    for name in cfg["groups"]:
        lo, hi = hues[name]
        in_band = (comps["hue"] >= lo) & (comps["hue"] < hi) if lo < hi else (comps["hue"] >= lo) | (comps["hue"] < hi)
        sel = big & (comps["chroma"] >= acfg["ink_chroma_min"]) & in_band
        if sel.sum() < cfg["min_components"]:
            stats[name] = {"n": int(sel.sum()), "bleed": False}
            continue
        sp = otsu_split(comps["L"][sel], comps["area"][sel])
        gap = sp["light_mean"] - sp["dark_mean"]
        lo_s, hi_s = cfg["light_share"]
        is_bleed = sp["eta"] >= cfg["min_eta"] and gap >= cfg["min_gap_L"] and lo_s <= sp["light_share"] <= hi_s
        stats[name] = {"n": int(sel.sum()), "eta": round(sp["eta"], 2), "gap": round(gap, 1),
                       "light_share": round(sp["light_share"], 2), "split_L": round(sp["threshold"], 1), "bleed": is_bleed}
        if is_bleed:
            light_ids.extend(np.flatnonzero(sel & (comps["L"] > sp["threshold"])).tolist())
    return lab, np.array(light_ids, dtype=np.int64), stats


def suppress_bleed(img: np.ndarray, groups: dict, dpi: float, cfg: dict, acfg: dict) -> tuple:
    """PIXEL-level: lighten pixels of a bleed colour group that are lighter than its split, away from dark ink of
    that colour and from black/grey ink. (Component-level classification merged bleed with touching front strokes
    and print, and erased them.) Returns (image, lightened mask)."""
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sk = mm_to_px(acfg["stroke_width_max_mm"], dpi, odd=True)
    bh = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_RECT, (sk, sk)))
    marks = bh > acfg["stroke_weak"]
    L = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)[..., 0]
    chroma = img.max(axis=2).astype(np.int16) - img.min(axis=2)
    hue = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[..., 0]
    coloured = chroma >= acfg["ink_chroma_min"]
    pk = mm_to_px(cfg["protect_mm"], dpi, odd=True)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (pk, pk))
    neutral_ink = (bh > acfg["stroke_ink"]) & ~coloured
    protect = cv2.dilate(neutral_ink.astype(np.uint8), kernel).astype(bool)
    mask = np.zeros(g.shape, bool)
    for name, st in groups.items():
        if not st.get("bleed"):
            continue
        lo, hi = acfg["ink_hues"][name]
        band = (hue >= lo) & (hue < hi) if lo < hi else (hue >= lo) | (hue < hi)
        group = marks & coloured & band
        dark = group & (L <= st["split_L"])
        light = group & (L > st["split_L"])
        protect_g = protect | cv2.dilate(dark.astype(np.uint8), kernel).astype(bool)
        mask |= light & ~protect_g
    if not mask.any():
        return img, mask
    alpha = np.clip(cv2.GaussianBlur(mask.astype(np.float32), (0, 0), max(0.5, cfg["feather_mm"] * dpi / 25.4)) * 1.5,
                    0, 1)[..., None] * cfg["lighten"]
    alpha = np.where(protect[..., None], 0.0, alpha)                         # print stays untouched, also at edges
    out = img.astype(np.float32) * (1 - alpha) + cfg["paper"] * alpha
    return np.clip(out + 0.5, 0, 255).astype(np.uint8), mask


def step9_bleed(rec: dict) -> None:
    if rec["metrics"].get(STEP9):                              # re-running the cell: start this step clean
        rec["flags"] = [f for f in rec["flags"] if f["step"] != STEP9]
        rec["actions"] = [a for a in rec["actions"] if a[0] != STEP9]
    cfg, review = CONFIG["bleed"], CONFIG["review"]["bleed"]
    img = rec["images"]["sharpened"]
    if rec["needs"]["skip_page"]:
        rec["images"]["bleed_clean"] = img
        rec["metrics"][STEP9] = {"applied": False}
        return
    t0 = time.perf_counter()
    lab, light_ids, stats = detect_bleed(img, rec["dpi"], cfg, CONFIG["assess"])
    step2_said = bool(rec["needs"]["bleed_suppression"])
    if light_ids.size == 0:
        rec["images"]["bleed_clean"] = img
        rec["metrics"][STEP9] = {"applied": False, "groups": stats, "step2_suspected": step2_said}
        add_action(rec, STEP9, "no bleed-through (no colour group splits into two lightness populations)")
        if step2_said:
            add_flag(rec, STEP9, "bleed_not_confirmed", "Step 2 suspected bleed-through but no clean lightness split: "
                                                         "not changed")
        return

    out, mask = suppress_bleed(img, stats, rec["dpi"], cfg, CONFIG["assess"])
    # front strokes = strokes that are not part of the lightened population; compare before/after
    keep = ~cv2.dilate(mask.astype(np.uint8), np.ones((3, 3), np.uint8)).astype(bool)
    f_before = stroke_pixels(np.where(keep[..., None], img, 255).astype(np.uint8), rec["dpi"],
                             CONFIG["denoise"]["stroke_min_mm2"], CONFIG["assess"])
    f_after = stroke_pixels(np.where(keep[..., None], out, 255).astype(np.uint8), rec["dpi"],
                            CONFIG["denoise"]["stroke_min_mm2"], CONFIG["assess"])
    front_change = (f_after - f_before) / max(1, f_before) * 100
    rec["images"]["bleed_clean"] = out
    groups = [g for g, s in stats.items() if s["bleed"]]
    rec["metrics"][STEP9] = {"applied": True, "groups": stats, "step2_suspected": step2_said,
                             "lightened_pct": round(float(mask.mean() * 100), 2),
                             "front_change_pct": round(front_change, 1),
                             "time_ms": round((time.perf_counter() - t0) * 1000)}
    add_action(rec, STEP9, f"lightened light {'/'.join(groups)} ink pixels ({mask.mean() * 100:.2f}% of page) "
                           f"{cfg['lighten']:.0%} toward paper; dark {'/'.join(groups)} and black/grey ink protected")
    add_flag(rec, STEP9, "bleed_suppressed", f"light {'/'.join(groups)} ink treated as show-through and lightened: "
                                             "check no real (lighter) writing was dimmed")
    if abs(front_change) > review["stroke_change_pct"]:
        add_flag(rec, STEP9, "ink_change", f"front stroke pixels {front_change:+.1f}%")


run_parallel(step9_bleed, page_records, "Step 9 complete")

In [ ]:
# ---------- Step 9 review: colour-group lightness split per page, what was lightened, zoom ----------
print(f"{'Page':<16}{'Step2?':>7}  Groups tested (n, eta, gap, light share)                     Applied  Flags")
for rec in page_records:
    m = rec["metrics"][STEP9]
    if "groups" not in m:
        continue
    g = "; ".join(f"{k}: n={v['n']}" + (f" eta={v['eta']} gap={v['gap']} share={v['light_share']}{' BLEED' if v['bleed'] else ''}"
                                        if "eta" in v else "") for k, v in m["groups"].items() if v["n"])
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP9) or "-"
    if m["applied"] or m["step2_suspected"] or any(v.get("eta", 0) >= 0.7 for v in m["groups"].values()):
        print(f"{page_label(rec):<16}{('yes' if m['step2_suspected'] else '-'):>7}  {g:<60}{('yes' if m['applied'] else '-'):>8}  {flags}")
applied9 = [r for r in page_records if r["metrics"][STEP9].get("applied")]
print(f"\nBleed-through lightened on {len(applied9)} / {len(page_records)} pages (all flagged for review). "
      "Pages with no colour group near the split are not listed.")
for rec in applied9:
    m = rec["metrics"][STEP9]
    print(f"  {page_label(rec)}: lightened {m['lightened_pct']}% of page, front strokes {m['front_change_pct']:+}%")
show_before_after(applied9, "sharpened", "bleed_clean", "Step 9: bleed-through (before | after)", STEP9)
show_zoom_pairs(applied9, "sharpened", "bleed_clean", "Step 9: bleed-through, zoomed where the change is largest",
                STEP9, crop_mm=60)

## Step 10: Contrast enhancement (faint ink)

**Why:** faint ballpoint, carbon copies and washed-out blue ink stay hard to read even on white paper.

**Detect on the current image:** `assess_page` on `images["bleed_clean"]`. Act only if the median grey of the strokes (`ink_level`) is ≥ 180.

**Method (adaptive tone curve on Lab L only):**
- Below a knee at L = 235: `L' = knee × (L / knee)^γ`. At and above the knee: unchanged.
- γ is chosen so the page's median ink reaches 140 (`target_ink_level`), capped at 2.5.
- Paper (255 after Step 6) stays exactly white. Near-white texture above the knee is not darkened. The curve is monotone (no reordering, no new edges). Colour (a/b) is untouched, so blue stays blue.
- *Not CLAHE:* the paper is already uniform after Step 6, and local histogram equalisation would bring back paper texture.

**Verify:** ink level, noise and texture before → after.

**Review flags:** `still_faint` (median ink still ≥ 170), `noise_amplified` (noise σ +0.5 or texture +0.5%).

In [ ]:
STEP10 = "step10_contrast"


def ink_gamma(ink_level: float, cfg: dict) -> float:
    knee = cfg["knee"]
    lvl = min(max(ink_level, 1.0), knee - 1.0)
    target = min(cfg["target_ink_level"], lvl)
    return float(np.clip(np.log(target / knee) / np.log(lvl / knee), 1.0, cfg["max_gamma"]))


def apply_tone_curve(img: np.ndarray, gamma: float, knee: float) -> np.ndarray:
    x = np.arange(256, dtype=np.float32)
    lut = np.where(x < knee, knee * (x / knee) ** gamma, x)
    lut = np.clip(lut + 0.5, 0, 255).astype(np.uint8)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.LUT(lab[..., 0], lut)
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def step10_contrast(rec: dict) -> None:
    if rec["metrics"].get(STEP10):                             # re-running the cell: start this step clean
        rec["flags"] = [f for f in rec["flags"] if f["step"] != STEP10]
        rec["actions"] = [a for a in rec["actions"] if a[0] != STEP10]
    cfg, review = CONFIG["contrast"], CONFIG["review"]["contrast"]
    img = rec["images"]["bleed_clean"]
    if rec["needs"]["skip_page"]:
        rec["images"]["contrast"] = img
        rec["metrics"][STEP10] = {"applied": False}
        return
    t0 = time.perf_counter()
    before = assess_page(img, CONFIG["assess"], rec["dpi"])
    if before["ink_level"] < CONFIG["assess"]["needs"]["faint_ink_level"]:
        rec["images"]["contrast"] = img
        rec["metrics"][STEP10] = {"applied": False, "ink_level": before["ink_level"]}
        add_action(rec, STEP10, f"ink dark enough (median {before['ink_level']})")
        return
    gamma = ink_gamma(before["ink_level"], cfg)
    out = apply_tone_curve(img, gamma, cfg["knee"])
    after = assess_page(out, CONFIG["assess"], rec["dpi"])
    rec["images"]["contrast"] = out
    rec["metrics"][STEP10] = {"applied": True, "gamma": round(gamma, 2),
                              "ink_level": [before["ink_level"], after["ink_level"]],
                              "noise_sigma": [before["noise_sigma"], after["noise_sigma"]],
                              "texture_pct": [before["texture_pct"], after["texture_pct"]],
                              "time_ms": round((time.perf_counter() - t0) * 1000)}
    add_action(rec, STEP10, f"tone curve gamma {gamma:.2f} below L {cfg['knee']}: median ink "
                            f"{before['ink_level']} -> {after['ink_level']}")
    if after["ink_level"] >= review["still_faint_level"]:
        add_flag(rec, STEP10, "still_faint", f"median ink still {after['ink_level']}")
    if (after["noise_sigma"] - before["noise_sigma"] > review["noise_rise_sigma"]
            or after["texture_pct"] - before["texture_pct"] > review["texture_rise_pct"]):
        add_flag(rec, STEP10, "noise_amplified", f"noise {before['noise_sigma']} -> {after['noise_sigma']}, "
                                                 f"texture {before['texture_pct']} -> {after['texture_pct']}%")


run_parallel(step10_contrast, page_records, "Step 10 complete")

In [ ]:
# ---------- Step 10 review: ink level of every page, what was enhanced, zoom ----------
lv = sorted(((r["metrics"][STEP10]["ink_level"] if not r["metrics"][STEP10].get("applied")
              else r["metrics"][STEP10]["ink_level"][0]), page_label(r))
            for r in page_records if "ink_level" in r["metrics"][STEP10])
print("Median ink grey before Step 10 (faintest first):", ", ".join(f"{l} {v}" for v, l in lv[::-1][:12]), "...")
applied10 = [r for r in page_records if r["metrics"][STEP10].get("applied")]
print(f"\n{'Page':<16}{'Gamma':>6}{'Ink grey':>14}{'Noise':>14}{'Texture %':>14}  Flags")
for rec in applied10:
    m = rec["metrics"][STEP10]
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP10) or "-"
    print(f"{page_label(rec):<16}{m['gamma']:>6}{m['ink_level'][0]:>7}->{m['ink_level'][1]:<5}"
          f"{m['noise_sigma'][0]:>7}->{m['noise_sigma'][1]:<6}{m['texture_pct'][0]:>7}->{m['texture_pct'][1]:<6}  {flags}")
print(f"\nEnhanced {len(applied10)} / {len(page_records)} pages")
step10_flagged = [r for r in page_records if any(f["step"] == STEP10 for f in r["flags"])]
print(f"Flagged for review in Step 10: {len(step10_flagged)}")
for rec in step10_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP10):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")
show_before_after(applied10, "bleed_clean", "contrast", "Step 10: faint-ink contrast (before | after)", STEP10)
show_zoom_pairs(applied10, "bleed_clean", "contrast", "Step 10: zoomed where the change is largest", STEP10, crop_mm=45)

## Step 11: Final outputs

Files are named after the source PDF and page number, so pages sort in order:

| File (in `output/<doc>/`) | What |
|---|---|
| `{doc}_{page}_original.jpg` | The **exact** embedded scan, byte for byte (`.png` if the page had to be rendered) |
| `{doc}_{page}_cleaned.png` | Final cleaned colour page |
| `{doc}_{page}_cleaned_bw.png` | 1-bit black & white (Sauvola) |
| `{doc}_{page}_params.json` | **What was applied to this page, in pipeline order:** each step with applied yes/no, why, the parameters actually used (angle, gain, NLM h, gamma, …), the result and its flags |
| `{doc}_{page}_stageNN_<stage>.jpg` | Every intermediate stage, only when `save_stages` / the device profile says so (GB10) |
| `{doc}_metrics.json` | All pages of the document: full metrics, actions, flags + the CONFIG and device used |
| `output/summary.json` | Per document: pages, pages needing review, flag counts |

The page number is zero-padded to at least 3 digits (`page_digits`), or more for longer documents.

**Black & white (Sauvola, NumPy/OpenCV only):**
- `T = mean × (1 + k × (std / R − 1))` over a 6 mm window (box filters on image and squared image), with k = 0.15 and R = 128.
- Computed on the **darkest channel** (min of B, G, R): faint blue ink is much darker in red than in luminance. Measured on all pages: 98.8% of stroke cores kept (median), versus 92.7% with luminance and k = 0.2. Noise stays ≤ 0.15%.
- Isolated black blobs < 0.02 mm² are removed.

**Verify the B&W against the colour page:**
- `bw_ink_recall`: share of stroke **cores** (black-hat > 45 inside real strokes) that are black in the B&W. Flag `bw_ink_loss` < 85%.
- `bw_noise`: black pixels farther than 0.5 mm from any stroke, % of page. Flag at > 1%.

In [ ]:
STEP11 = "step11_output"
STAGES = ["original", "oriented", "borders_clean", "deskewed", "illum_norm", "denoised", "sharpened", "bleed_clean", "contrast"]


def sauvola_bw(img: np.ndarray, dpi: float, cfg: dict) -> np.ndarray:
    """Sauvola adaptive threshold (white paper = 255, ink = 0), small isolated blobs removed."""
    g = (img.min(axis=2) if cfg["source"] == "min_channel" else cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)).astype(np.float32)
    w = mm_to_px(cfg["window_mm"], dpi, odd=True)
    mean = cv2.boxFilter(g, -1, (w, w), borderType=cv2.BORDER_REFLECT)
    sq = cv2.boxFilter(g * g, -1, (w, w), borderType=cv2.BORDER_REFLECT)
    std = np.sqrt(np.maximum(sq - mean * mean, 0))
    thr = mean * (1 + cfg["k"] * (std / cfg["R"] - 1))
    ink = (g <= thr).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink, connectivity=8)
    small = st[:, cv2.CC_STAT_AREA] < cfg["min_blob_mm2"] * (dpi / 25.4) ** 2
    small[0] = False
    ink[small[lab]] = 0
    return np.where(ink > 0, 0, 255).astype(np.uint8)


def bw_checks(color: np.ndarray, bw: np.ndarray, dpi: float) -> dict:
    """How much of the real ink is black in the B&W, and how much black is far from any ink."""
    acfg = CONFIG["assess"]
    g = cv2.cvtColor(color, cv2.COLOR_BGR2GRAY)
    sk = mm_to_px(acfg["stroke_width_max_mm"], dpi, odd=True)
    bh = cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_RECT, (sk, sk)))
    ink = (bh > acfg["stroke_ink"]).astype(np.uint8)
    _, lab, st, _ = cv2.connectedComponentsWithStats(ink, connectivity=8)
    real = st[:, cv2.CC_STAT_AREA] >= CONFIG["denoise"]["stroke_min_mm2"] * (dpi / 25.4) ** 2
    real[0] = False
    strokes = real[lab] & (bh > acfg["stroke_strong"])        # cores only: soft anti-aliased edges may go white
    black = bw == 0
    near = cv2.dilate((bh > acfg["stroke_weak"]).astype(np.uint8),
                      np.ones((mm_to_px(0.5, dpi, odd=True),) * 2, np.uint8)).astype(bool)
    return {"bw_ink_recall": round(float(black[strokes].mean()) if strokes.any() else 1.0, 3),
            "bw_noise_pct": round(float((black & ~near).mean() * 100), 3),
            "bw_black_pct": round(float(black.mean() * 100), 2)}


def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return o


def page_stem(rec: dict) -> str:
    """{original file name}_{page number}, page zero-padded so files sort in page order."""
    digits = max(CONFIG["output"]["page_digits"], len(str(rec.get("n_pages") or 0)))
    return f"{rec['doc']}_{rec['page_no']:0{digits}d}"


def applied_steps(rec: dict) -> list:
    """Every step in pipeline order: applied or not, why, the parameters actually used, and the result."""
    m, C = rec["metrics"], CONFIG
    why = lambda step: next((a for s, a in reversed(rec["actions"]) if s == step), "")
    flags = lambda step: [f["code"] for f in rec["flags"] if f["step"] == step]
    out = []

    def add(step, name, applied, params, result):
        out.append({"order": len(out) + 1, "step": name, "applied": bool(applied), "why": why(step),
                    "params": params if applied else {}, "result": result, "flags": flags(step)})

    s1 = m.get(STEP1, {})
    add(STEP1, "render", True, {"method": s1.get("method"), "codec": s1.get("codec"), "dpi_used": s1.get("dpi_used"),
                                "upscale": s1.get("upscale"), **({"fallback_dpi": C["render"]["fallback_dpi"]}
                                                                 if s1.get("method") == "rendered" else {})},
        {k: s1.get(k) for k in ("native_px", "output_px", "orientation", "greyscale_scan", "paper_tint")})
    add(STEP2, "assess (measure only)", False, {}, {"needs": rec.get("needs")})
    s3 = m.get(STEP3, {})
    add(STEP3, "orientation", s3.get("rotation", 0) != 0, {"rotation_deg_cw": s3.get("rotation")},
        {k: s3.get(k) for k in ("decision", "axis_score", "updown_score")})
    s4 = m.get(STEP4, {})
    add(STEP4, "border cleanup", s4.get("applied"), {**C["borders"], "fill_bgr": s4.get("fill_bgr")},
        {k: s4.get(k) for k in ("sides", "filled_pct", "shadow_flattened_pct", "restored_mm2", "beyond_spine_ink_mm2",
                                "dark_kept_original_pct", "inverted_glyphs", "remaining_after")})
    s5 = m.get(STEP5, {})
    add(STEP5, "deskew", s5.get("rotated"), {"angle_deg": s5.get("angle"), "source": s5.get("source"),
                                              "interpolation": "cubic", "canvas": "expanded"},
        {k: s5.get(k) for k in ("line_angle", "n_lines", "projection_angle", "projection_peak")})
    s6 = m.get(STEP6, {})
    add(STEP6, "illumination & paper colour", s6.get("applied"), dict(C["illumination"]),
        {k: s6.get(k) for k in ("before", "after", "gain_capped_pct", "dark_object_pct", "ink_change_pct")})
    s7 = m.get(STEP7, {})
    p7 = s7.get("plan", {})
    add(STEP7, "noise removal", s7.get("applied"),
        {"nlm": p7.get("nlm"), "nlm_h": C["denoise"]["nlm_h"].get(p7.get("nlm")),
         "nlm_template_mm": C["denoise"]["nlm_template_mm"], "nlm_search_mm": C["denoise"]["nlm_search_mm"],
         "chroma_median_mm": C["denoise"]["chroma_median_mm"] if p7.get("chroma") else None,
         "texture_cleanup": p7.get("texture"), "speck_removal": p7.get("specks")},
        {k: s7.get(k) for k in ("before", "after", "ink_change_pct", "texture_px_cleared", "speck_px_cleared")})
    s8 = m.get(STEP8, {})
    add(STEP8, "sharpen", s8.get("applied"), {"amount": s8.get("amount"), **{k: C["sharpen"][k] for k in
                                               ("sigma_mm", "detail_threshold", "halo_overshoot")}},
        {k: s8.get(k) for k in ("sharpness", "noise_sigma", "stroke_change_pct")})
    s9 = m.get(STEP9, {})
    add(STEP9, "bleed-through", s9.get("applied"), {k: C["bleed"][k] for k in ("lighten", "protect_mm", "min_eta",
                                                     "min_gap_L", "light_share")},
        {k: s9.get(k) for k in ("groups", "lightened_pct", "front_change_pct")})
    s10 = m.get(STEP10, {})
    add(STEP10, "contrast (faint ink)", s10.get("applied"), {"gamma": s10.get("gamma"), "knee": C["contrast"]["knee"]},
        {k: s10.get(k) for k in ("ink_level", "noise_sigma", "texture_pct")})
    s11 = m.get(STEP11, {})
    add(STEP11, "black & white", True, dict(C["output"]["bw"]),
        {k: s11.get(k) for k in ("bw_ink_recall", "bw_noise_pct", "bw_black_pct")})
    return out


def step11_output(rec: dict) -> None:
    if rec["metrics"].get(STEP11):                             # re-running the cell: start this step clean
        rec["flags"] = [f for f in rec["flags"] if f["step"] != STEP11]
        rec["actions"] = [a for a in rec["actions"] if a[0] != STEP11]
    out_cfg, review = CONFIG["output"], CONFIG["review"]["output"]
    doc_dir = OUTPUT_DIR / rec["doc"]
    doc_dir.mkdir(parents=True, exist_ok=True)
    stem = page_stem(rec)
    color = rec["images"]["contrast"]
    bw = sauvola_bw(color, rec["dpi"], out_cfg["bw"])
    rec["images"]["clean_color"], rec["images"]["clean_bw"] = color, bw

    comp = [cv2.IMWRITE_PNG_COMPRESSION, out_cfg["png_compression"]]
    if rec.get("original_bytes"):                              # the exact embedded scan, byte for byte
        orig_path = doc_dir / f"{stem}_original.jpg"
        orig_path.write_bytes(rec["original_bytes"])
    else:                                                      # page was rendered (not a single JPEG scan)
        orig_path = doc_dir / f"{stem}_original.png"
        cv2.imwrite(str(orig_path), rec["images"]["original"], comp)
    color_path, bw_path = doc_dir / f"{stem}_cleaned.png", doc_dir / f"{stem}_cleaned_bw.png"
    cv2.imwrite(str(color_path), color, comp)
    cv2.imwrite(str(bw_path), bw, comp + [cv2.IMWRITE_PNG_BILEVEL, 1])
    files = {"original": orig_path.name, "cleaned": color_path.name, "cleaned_bw": bw_path.name}

    save_stages = PROFILE["save_intermediates"] if out_cfg["save_stages"] is None else out_cfg["save_stages"]
    if save_stages:
        for k, stage in enumerate(STAGES):
            p = doc_dir / f"{stem}_stage{k:02d}_{stage}.jpg"
            cv2.imwrite(str(p), rec["images"][stage], [cv2.IMWRITE_JPEG_QUALITY, 90])
            files[f"stage{k:02d}_{stage}"] = p.name

    checks = bw_checks(color, bw, rec["dpi"]) if not rec["needs"]["skip_page"] else {}
    rec["metrics"][STEP11] = {"files": files, "size_px": [int(color.shape[1]), int(color.shape[0])], **checks}
    add_action(rec, STEP11, f"saved {orig_path.name}, {color_path.name}, {bw_path.name}" + (" + stages" if save_stages else ""))
    if checks and checks["bw_ink_recall"] < review["bw_ink_recall"]:
        add_flag(rec, STEP11, "bw_ink_loss", f"only {checks['bw_ink_recall']:.0%} of stroke cores are black in the B&W")
    if checks and checks["bw_noise_pct"] > review["bw_noise_pct"]:
        add_flag(rec, STEP11, "bw_noise", f"{checks['bw_noise_pct']}% of the page is black away from any stroke")

    params_path = doc_dir / f"{stem}_params.json"             # what was applied to this page, in pipeline order
    rec["metrics"][STEP11]["files"]["params"] = params_path.name
    with open(params_path, "w", encoding="utf-8") as fh:
        json.dump(to_jsonable({"document": rec["doc"], "source": rec["doc_path"], "page": rec["page_no"],
                               "dpi": round(rec["dpi"], 1), "device": PROFILE["name"], "files": files,
                               "needs_review": bool(rec["flags"]), "flags": rec["flags"],
                               "steps": applied_steps(rec)}), fh, ensure_ascii=False, indent=1)


def write_reports(records: list[dict]) -> dict:
    """{doc}/{doc}_metrics.json per document (full metrics, pages in order) + summary.json across documents."""
    summary = {"generated": time.strftime("%Y-%m-%d %H:%M:%S"), "device": PROFILE["name"], "documents": {}}
    for doc_name in dict.fromkeys(r["doc"] for r in records):
        recs = sorted((r for r in records if r["doc"] == doc_name), key=lambda r: r["page_no"])
        pages = [to_jsonable({"page_no": r["page_no"], "dpi": round(r["dpi"], 1), "needs": r["needs"],
                              "metrics": r["metrics"], "actions": [{"step": s, "action": a} for s, a in r["actions"]],
                              "flags": r["flags"], "needs_review": bool(r["flags"])}) for r in recs]
        with open(OUTPUT_DIR / doc_name / f"{doc_name}_metrics.json", "w", encoding="utf-8") as fh:
            json.dump({"document": doc_name, "source": recs[0]["doc_path"], "pages": pages,
                       "config": to_jsonable(CONFIG), "device": to_jsonable({k: PROFILE[k] for k in
                       ("name", "system", "opencv", "backends", "num_workers")})}, fh, ensure_ascii=False, indent=1)
        counts = {}
        for r in recs:
            for f in r["flags"]:
                counts[f["code"]] = counts.get(f["code"], 0) + 1
        summary["documents"][doc_name] = {"pages": len(recs), "pages_needing_review": [r["page_no"] for r in recs if r["flags"]],
                                          "flag_counts": dict(sorted(counts.items(), key=lambda kv: -kv[1]))}
    with open(OUTPUT_DIR / "summary.json", "w", encoding="utf-8") as fh:
        json.dump(summary, fh, ensure_ascii=False, indent=1)
    return summary


run_parallel(step11_output, page_records, "Step 11 images saved")

write_reports(page_records)
print(f"Step 11 complete: per page {{doc}}_{{page}}_original/_cleaned/_cleaned_bw/_params.json, "
      f"per document {{doc}}_metrics.json, summary.json in {OUTPUT_DIR.resolve()}")

In [ ]:
# ---------- Step 11 review: B&W checks, files written, final colour vs B&W ----------
print(f"{'Page':<16}{'Ink recall':>11}{'BW noise %':>11}{'Black %':>9}  Flags")
for rec in page_records:
    m = rec["metrics"][STEP11]
    if "bw_ink_recall" not in m:
        continue
    flags = ", ".join(f["code"] for f in rec["flags"] if f["step"] == STEP11) or "-"
    if flags != "-" or m["bw_ink_recall"] < 0.97:
        print(f"{page_label(rec):<16}{m['bw_ink_recall']:>11}{m['bw_noise_pct']:>11}{m['bw_black_pct']:>9}  {flags}")
recalls = [r["metrics"][STEP11]["bw_ink_recall"] for r in page_records if "bw_ink_recall" in r["metrics"][STEP11]]
noises = [r["metrics"][STEP11]["bw_noise_pct"] for r in page_records if "bw_noise_pct" in r["metrics"][STEP11]]
print(f"(pages with recall >= 0.97 and no flag not listed) | recall median {np.median(recalls):.3f}, min {min(recalls):.3f} | "
      f"B&W noise median {np.median(noises):.3f}%, max {max(noises):.3f}%")
step11_flagged = [r for r in page_records if any(f["step"] == STEP11 for f in r["flags"])]
print(f"Flagged for review in Step 11: {len(step11_flagged)}")
for rec in step11_flagged:
    for f in (f for f in rec["flags"] if f["step"] == STEP11):
        print(f"  ⚑ {page_label(rec)}: [{f['code']}] {f['message']}")

n_files = sum(len(r["metrics"][STEP11]["files"]) for r in page_records)
size_mb = sum((OUTPUT_DIR / r["doc"] / p).stat().st_size for r in page_records for p in r["metrics"][STEP11]["files"].values()) / 1024**2
print(f"\nFiles written: {n_files} ({size_mb:.0f} MB) under {OUTPUT_DIR.resolve()}")
with open(OUTPUT_DIR / "summary.json", encoding="utf-8") as fh:
    for doc_name, d in json.load(fh)["documents"].items():
        print(f"  {doc_name}: {d['pages']} pages, {len(d['pages_needing_review'])} need review {d['pages_needing_review']}")

sample = step11_flagged + [r for r in page_records if r not in step11_flagged][::12]
show_before_after([r for r in sample if not r["needs"]["skip_page"]], "clean_color", "clean_bw",
                  "Step 11: final colour | final black & white", STEP11, max_pages=9)

## Batch mode (large document sets)

The step-by-step cells above keep every stage of every page in memory, about 100 MB per page, so they are meant for review on a few documents. For large sets, `process_documents()` streams each PDF:
1. One quick pass reads the image sizes only, to get the document's DPI (median).
2. Pages are rendered in page order and processed `pages_in_flight` at a time on the worker threads (Steps 2–11, the same functions as above).
3. Each page's files are written, then its images are freed. Only the metrics stay in memory.

Output files, names and order are identical to the step-by-step run.

In [ ]:
PIPELINE = [step2_assess, step3_orient, step4_borders, step5_deskew, step6_illumination, step7_denoise,
            step8_sharpen, step9_bleed, step10_contrast]


def process_page(rec: dict, write: bool = True) -> dict:
    """Steps 2-10 (and Step 11 = write files) on one rendered page record, in pipeline order."""
    for step in PIPELINE:
        step(rec)
    if write:
        step11_output(rec)
    return rec


def document_dpi(doc) -> float | None:
    """Median DPI estimate of a document from the embedded scan sizes only (no decoding)."""
    est = []
    for i in range(len(doc)):
        scan, _ = find_page_scan(doc[i])
        if scan is not None:
            w, h = scan.get_px_size()
            est.append(min(w, h) / CONFIG["render"]["assumed_paper_width_in"])
    return float(np.median(est)) if est else None


def process_documents(pdf_paths: list, keep_images: bool | None = None) -> list[dict]:
    """Stream every page of every PDF through the pipeline; returns the page records (images freed)."""
    keep = CONFIG["batch"]["keep_images"] if keep_images is None else keep_images
    in_flight = CONFIG["batch"]["pages_in_flight"] or 2 * PROFILE["num_workers"]
    records, t0 = [], time.perf_counter()
    for pdf_path in map(Path, pdf_paths):
        doc = pdfium.PdfDocument(str(pdf_path))
        n = len(doc)
        doc_dpi = None if CONFIG["render"]["dpi_override"] or not CONFIG["render"]["dpi_per_document"] else document_dpi(doc)
        for start in range(0, n, in_flight):
            chunk = []
            for i in range(start, min(n, start + in_flight)):          # pdfium is not thread-safe: render here
                rec = new_page_record(pdf_path.stem, pdf_path, i, doc[i], n)
                rec["images"]["original"] = render_page(doc[i], rec, CONFIG["render"], CONFIG["review"]["render"],
                                                        PROFILE["max_render_scale"])
                if doc_dpi:
                    rec["dpi"] = doc_dpi * rec["metrics"][STEP1]["upscale"]
                    rec["metrics"][STEP1]["dpi_used"] = round(rec["dpi"])
                chunk.append(rec)
            with ThreadPoolExecutor(max_workers=PROFILE["num_workers"]) as pool:
                list(pool.map(process_page, chunk))
            for rec in chunk:
                if not keep:
                    rec["images"] = {}
                    rec.pop("original_bytes", None)
            records.extend(chunk)
            print(f"  {pdf_path.stem}: pages {start + 1}-{min(n, start + in_flight)} / {n} done "
                  f"({time.perf_counter() - t0:.0f}s)")
        doc.close()
    summary = write_reports(records)
    print(f"Batch complete: {len(records)} pages in {time.perf_counter() - t0:.0f}s -> {OUTPUT_DIR.resolve()}")
    for d, s in summary["documents"].items():
        print(f"  {d}: {s['pages']} pages, {len(s['pages_needing_review'])} need review")
    return records


# Example (large sets): process every PDF in a folder without keeping images in memory
RUN_BATCH = False
if RUN_BATCH:
    batch_records = process_documents(sorted(Path(CONFIG["input"]["pdf_dir"]).glob("*.pdf")))

## Tests: information preservation

Planted "content traps", each something a real FIR file can contain, are pasted onto a real clean page. The page then runs through the whole pipeline (Steps 2–10, nothing written), and each trap is checked. A failing test means a step removed information it must keep.

| Trap | Must survive |
|---|---|
| Black banner with **white text** at the top edge (looks like a dark border) | white text still readable on dark (text-vs-banner contrast ≥ 60% of the original) |
| Dark **photo** block 25 × 30 mm (a dark object wider than the background window) | mean tone within 15% and detail (std) ≥ 80% of the original |
| **Faint purple stamp** (grainy dots, light) | ≥ 80% of its coloured pixels still coloured |
| **Faint pencil** strokes (light grey, L ≈ 215) | still visible: ≥ 80% of the stroke pixels darker than 235 |
| **Thin blue line** (1 px) | keeps its colour: chroma ≥ 60% of the original |
| **Light blue writing only** (no dark blue on the page) | not treated as bleed-through by Step 9 |

The pipeline also runs a second time on the same page to confirm the results are reproducible (deterministic).

In [ ]:
def make_trap_page(base: dict) -> tuple:
    """Copy of a real page record with planted content traps. Returns (record, {trap: (x0, y0, x1, y1)})."""
    img = base["images"]["original"].copy()
    dpi = base["dpi"]
    px = lambda mm: mm_to_px(mm, dpi)
    h, w = img.shape[:2]
    rng = np.random.default_rng(7)
    boxes = {}
    # 1. black banner with white text at the top edge
    x0, y0, x1, y1 = px(15), px(2), w - px(15), px(2) + px(9)
    img[y0:y1, x0:x1] = 25
    cv2.putText(img, "MEDICO LEGAL REPORT 2024", (x0 + px(5), y1 - px(2)), cv2.FONT_HERSHEY_SIMPLEX,
                dpi / 200 * 1.6, (245, 245, 245), max(2, px(0.6)), cv2.LINE_AA)
    boxes["white_on_black_banner"] = (x0, y0, x1, y1)
    # 2. dark photo block with detail
    x0, y0 = px(25), h // 2 - px(15)
    x1, y1 = x0 + px(25), y0 + px(30)
    yy, xx = np.mgrid[0:y1 - y0, 0:x1 - x0]
    photo = (60 + 50 * np.sin(xx / px(3)) * np.cos(yy / px(4)) + rng.normal(0, 12, xx.shape)).clip(0, 255)
    img[y0:y1, x0:x1] = photo[..., None].astype(np.uint8)
    boxes["dark_photo"] = (x0, y0, x1, y1)
    # 3. faint purple stamp: grainy ring of dots
    cx, cy, r = w - px(45), h // 2, px(14)
    stamp = np.zeros((h, w), np.uint8)
    cv2.circle(stamp, (cx, cy), r, 255, px(1.2))
    cv2.putText(stamp, "SEAL", (cx - px(9), cy + px(2)), cv2.FONT_HERSHEY_SIMPLEX, dpi / 200 * 1.0, 255, px(0.5))
    stamp &= (rng.random((h, w)) < 0.55).astype(np.uint8) * 255
    img[stamp > 0] = (200, 150, 200)
    boxes["faint_purple_stamp"] = (cx - r - px(2), cy - r - px(2), cx + r + px(2), cy + r + px(2))
    # 4. faint pencil strokes
    x0, y0 = px(25), h - px(60)
    for k in range(4):
        cv2.line(img, (x0, y0 + k * px(4)), (x0 + px(50), y0 + k * px(4) + px(2)), (215, 215, 215), max(1, px(0.25)))
    boxes["faint_pencil"] = (x0, y0 - px(1), x0 + px(50), y0 + px(16))
    # 5. thin blue line
    x0, y0 = w // 2 - px(30), h - px(35)
    cv2.line(img, (x0, y0), (x0 + px(60), y0), (190, 90, 30), 1)
    boxes["thin_blue_line"] = (x0, y0 - 2, x0 + px(60), y0 + 3)
    # 6. light blue writing only (no dark blue)
    x0, y0 = w // 2 - px(40), h - px(25)
    for k in range(3):
        cv2.putText(img, "light blue note", (x0, y0 + k * px(6)), cv2.FONT_HERSHEY_SCRIPT_SIMPLEX,
                    dpi / 200 * 1.1, (230, 190, 150), max(1, px(0.35)), cv2.LINE_AA)
    boxes["light_blue_only"] = (x0, y0 - px(5), x0 + px(80), y0 + px(14))

    rec = {**{k: v for k, v in base.items() if k not in ("images", "metrics", "flags", "actions", "needs")},
           "doc": base["doc"] + "_TRAP", "images": {"original": img}, "metrics": {STEP1: dict(base["metrics"][STEP1])},
           "flags": [], "actions": []}
    return rec, boxes


def trap_checks(before: np.ndarray, after: np.ndarray, boxes: dict, rec: dict) -> dict:
    """Map each trap box from the original page into the final page (orientation/deskew can move it)."""
    res = {}
    s5 = rec["metrics"][STEP5]
    if s5.get("rotated"):                                      # undo the deskew so both images share coordinates
        hh, ww = before.shape[:2]
        M = cv2.getRotationMatrix2D((ww / 2, hh / 2), s5["angle"], 1.0)
        M[0, 2] += after.shape[1] / 2 - ww / 2
        M[1, 2] += after.shape[0] / 2 - hh / 2
        after = cv2.warpAffine(after, cv2.invertAffineTransform(M), (ww, hh), flags=cv2.INTER_LINEAR,
                               borderValue=(255, 255, 255))

    def region(img, box):
        x0, y0, x1, y1 = box
        return img[y0:y1, x0:x1]
    L = lambda im: cv2.cvtColor(im, cv2.COLOR_BGR2LAB)[..., 0].astype(np.float32)
    chroma = lambda im: (im.max(axis=2).astype(np.int16) - im.min(axis=2)).astype(np.float32)

    b, a = region(before, boxes["white_on_black_banner"]), region(after, boxes["white_on_black_banner"])
    bl, al = L(b), L(a)
    text = bl > 150
    c_b = bl[text].mean() - bl[~text].mean()
    c_a = al[text].mean() - al[~text].mean() if text.any() else 0
    res["white_on_black_banner"] = (c_a >= 0.6 * c_b, f"contrast {c_b:.0f} -> {c_a:.0f}")

    b, a = L(region(before, boxes["dark_photo"])), L(region(after, boxes["dark_photo"]))
    ok = abs(a.mean() - b.mean()) <= 0.15 * b.mean() and a.std() >= 0.8 * b.std()
    res["dark_photo"] = (ok, f"mean {b.mean():.0f} -> {a.mean():.0f}, std {b.std():.0f} -> {a.std():.0f}")

    b, a = region(before, boxes["faint_purple_stamp"]), region(after, boxes["faint_purple_stamp"])
    cb = chroma(b) >= 30
    kept = float((chroma(a)[cb] >= 20).mean()) if cb.any() else 1.0
    res["faint_purple_stamp"] = (kept >= 0.8, f"{kept:.0%} of coloured stamp pixels still coloured")

    b, a = L(region(before, boxes["faint_pencil"])), L(region(after, boxes["faint_pencil"]))
    strokes = b < 230
    vis = float((a[strokes] < 235 * 100 / 100).mean()) if strokes.any() else 1.0
    res["faint_pencil"] = (vis >= 0.8, f"{vis:.0%} of pencil pixels still visible (L < 235)")

    b, a = region(before, boxes["thin_blue_line"]), region(after, boxes["thin_blue_line"])
    line = chroma(b) >= 40
    kept = float(chroma(a)[line].mean() / chroma(b)[line].mean()) if line.any() else 1.0
    res["thin_blue_line"] = (kept >= 0.6, f"chroma kept {kept:.0%}")

    res["light_blue_only"] = (not rec["metrics"][STEP9].get("applied"), "Step 9 applied" if rec["metrics"][STEP9].get("applied")
                              else "not treated as bleed-through")
    return res


RUN_TESTS = True
if RUN_TESTS:
    base_label = "FIR-4-2013 p4"                              # a clean printed form page
    base = next((r for r in page_records if page_label(r) == base_label),
                next((r for r in page_records if not r["needs"]["skip_page"]), page_records[0]))
    trap, boxes = make_trap_page(base)
    process_page(trap, write=False)
    results = trap_checks(trap["images"]["original"], trap["images"]["contrast"], boxes, trap)
    print(f"Information-preservation tests on {page_label(base)} (+ planted traps):")
    for name, (ok, msg) in results.items():
        print(f"  {'PASS' if ok else 'FAIL'}  {name:<24} {msg}")
    print("  flags raised on the trap page:", sorted({f['code'] for f in trap['flags']}))
    # determinism: same input -> same output
    trap2, _ = make_trap_page(base)
    process_page(trap2, write=False)
    same = np.array_equal(trap["images"]["contrast"], trap2["images"]["contrast"])
    print(f"  {'PASS' if same else 'FAIL'}  deterministic             identical output on a second run")
    show_before_after([trap], "original", "contrast", "Test page: planted traps (before | after)", STEP4, max_pages=1, h=900)
    assert all(ok for ok, _ in results.values()) and same, "information-preservation test failed"